# TIB PCB Commissioning

Measure what **working** means for this assembled PCB: routes and connector identity, levels and losses,
calibrated range, noise, response, laser current granularity, and throughput behavior. Run each experiment
individually in the lab and chamber. This is an editable scientific notebook, not an automatic acceptance test.

New code lives here; `hispec_fibpcb.py` owns the command interface, collectors, and firmware-model evaluation.
The original [attenuator lab](attenuator_calibration_lab.ipynb), [throughput lab](throuput_monitor_lab.ipynb),
and [combined attenuation/throughput lab](atten_and_tput_cal_and_noise_lab.ipynb) remain intact. Electrical scope work is in
[tib_fvoa_noise_scope_lab.ipynb](tib_fvoa_noise_scope_lab.ipynb), which runs independently.

Use the workspace `.venv` kernel and run experiments individually. This edited lab copy retains your enabled
command cells: **Run All can operate hardware**. For offline review, execute the imports, table/analysis
definitions, reference tables, and selected replay cells only; no connection is needed. There is no prescribed total
runtime or guaranteed warmup time; keep transients and decide later which samples are stationary.

1. Configuration, inventory, board operation, dark
2. Physical routing and assembled static losses
3. Embedded autocalibration and combined fit figures
4. Adjacent current steps and independent fixed-current holds
5. Noise, response, throughput, and room/chamber/build comparisons

Firmware references: [commands](../doc/commands.md), [hardware](../doc/hardware.md),
[calibration](../doc/attenuator_calibration.md), [PD notes](../doc/photodiode_notes.md).


In [1]:
from pathlib import Path
from dataclasses import asdict, is_dataclass, fields
from datetime import datetime, timezone
import math
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import display

TOOLS = next(p.resolve() for p in (Path.cwd(), Path.cwd() / 'tools', Path.cwd() / 'hispec-tib/tools')
             if (p / 'hispec_fibpcb.py').is_file())
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))
import hispec_fibpcb as hspcb

plt.rcParams.update({'axes.grid': True, 'grid.alpha': 0.18, 'figure.figsize': (11, 5)})

In [ ]:
CHANNEL = 'yj'                         # Change to 'hk' for the other channel.
LASERS = {'yj': ['1028y', '1270j', '1430yj'], 'hk': ['1430hk', '1510h', '2330k']}[CHANNEL]
BROKER, DEVICE = 'hispec.caltech.edu', 'hsfib-tib'
OUTPUT, FIBER = f'{CHANNEL}_ao', 'M'    # Match the physical patch before each experiment.
CONDITION = 'room'                     # e.g. 'room', 'chamber_-5C'; measured temperatures are also saved.
BUILD_LABEL = 'attenuator first autolevel' # Built with attenuator first or laser first autoleveling used for bookkeeping, not code
STARTUP_HISTORY = 'enter cold bank / previously enabled / elapsed idle time'   #used for bookkeeping, not code
PATCH_NOTES = 'SW3 C FEI->5dB->FFLC MM Green; SW3 B AO->3dB->FFLC MM Yellow' # used for bookkeeping, not code: enter actual fiber endpoints, jumpers, fixed attenuators, and meters'
FLASHED_ADC_SPS = 64                   # Flashed setup, not queried by the current API.
DETECTOR_BANDWIDTH_HZ = {'yj': 20.0, 'hk': 500.0}[CHANNEL]
ADC_INPUT_CAPACITANCE_UF = 0.47
CONNECT = True
pcb = globals().get('pcb')

DATA_DIR = TOOLS / 'commissioning_data'
CAPTURE_FILES = globals().get('CAPTURE_FILES', [])
CALIBRATION_FILES = globals().get('CALIBRATION_FILES', {})
ROUTE_FILES = globals().get('ROUTE_FILES', [])
EXTERNAL_SETUP = PATCH_NOTES            # Initial default; edit the matching patches row for later captures.
EXTERNAL_LOSS_DB = np.nan               # Unknown is allowed; never silently treated as zero.
EXTERNAL_LOSS_BASIS = 'unknown'          # 'measured', 'estimated', or 'unknown'; retain the description above.


## Configuration and archives

Every new archive is a compressed NPZ with named arrays and scalar-column tables. `load_tables(path)` returns
DataFrames; `context` is a typed settings table (`key`, `kind`, `number`, `text`). Raw samples, raw calibration
records, fits, bridge anchors, timestamps, temperatures, conditions, and errors are retained. No new JSON or
pickle payloads. The only JSON reader below opens existing calibration archives from the earlier lab.

The installed build is not inferred from the repository: enter its label above. Compare the actual flashed
attenuation-first and laser-first policies using those recorded labels. The owner-specified 64 SPS ADC, YJ 20 Hz detector, HK 500 Hz detector, and 0.47 µF ADC-input
capacitor are recorded as setup assumptions. Capacitance alone does not specify the analog cutoff.

## Measured laser-power references

These are the supplied **S154C** readings in µW at fractional laser command values 0.1, 0.5, and 1.0.
The sensor was reached through the laser's static attenuator and a measurement fiber of **unknown loss**.
The 25/50 dB values identify the attenuator parts; actual attenuation is wavelength dependent.
Reported ± values are retained without assuming standard deviation, error of the mean, or sensor accuracy.
In particular, 0.02308 ± 0.000013 µW is 23.08 nW ± 13 pW.

Edit these results directly; there is no meter-acquisition sequence here. The nominal diode comparison uses
[laser_properties.h](../app/src/laser_properties.h), copied below for independent offline analysis. It assumes
the default untuned current model, not a measurement of actual current. The apparent model-to-sensor attenuation
includes diode-model error, the static attenuator, and the measurement cable. It does not identify their losses
separately and is not a firmware property update. Route analysis uses exact measured levels only; another level
has no meter reference until one is entered. HK has no supplied meter readings.


In [ ]:
# Compiled nominal comparison only; live captures retain their queried settings separately.
laser_nominal = pd.DataFrame([
    ('1028y', 1028.01, 14.5, 250., .185), ('1270j', 1270., 8., 60., .166),
    ('1430yj', 1430., 8., 60., .166), ('1430hk', 1430., 8., 60., .166),
    ('1510h', 1510., 8., 60., .166), ('2330k', 2329.81, 24.9, 120., .031)],
    columns=['laser', 'wavelength_nm', 'threshold_ma', 'nominal_ma', 'efficiency_mw_per_ma']).set_index('laser')
source_power_readings = pd.DataFrame([
    ('1028y', .1, 3.347, .078, 25.), ('1028y', .5, 17.87, .350, 25.), ('1028y', 1., 35.25, .855, 25.),
    ('1270j', .1, 1.469, .002, 25.), ('1270j', .5, 11.30, .0034, 25.), ('1270j', 1., 23.600, .0086, 25.),
    ('1430yj', .1, .02308, .000013, 50.), ('1430yj', .5, .07619, .000040, 50.),
    ('1430yj', 1., .1392, .000072, 50.)],
    columns=['laser', 'level', 'power_uw', 'reported_pm_uw', 'static_part_label_db'])
source_power_readings['sensor'] = 'S154C'
source_power_readings['reference_plane'] = 'sensor after static attenuator and measurement fiber'
source_power_readings['measurement_fiber_loss_db'] = np.nan
source_power_readings['uncertainty_kind'] = 'reported ±; statistical interpretation unspecified'
display(source_power_readings[['laser', 'level', 'power_uw', 'reported_pm_uw', 'static_part_label_db']])


In [ ]:
source_comparison = source_power_readings.merge(laser_nominal, left_on='laser', right_index=True)
source_comparison['nominal_diode_uw'] = (1000 * source_comparison.level *
    (source_comparison.nominal_ma-source_comparison.threshold_ma) * source_comparison.efficiency_mw_per_ma)
source_comparison['apparent_combined_db'] = 10*np.log10(source_comparison.nominal_diode_uw/source_comparison.power_uw)
display(source_comparison[['laser', 'level', 'nominal_diode_uw', 'power_uw', 'apparent_combined_db']])
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), layout='constrained')
for ax, (name, group) in zip(axes, source_comparison.groupby('laser', sort=False)):
    ax.errorbar(group.level, group.power_uw, yerr=group.reported_pm_uw, fmt='o', capsize=3, label='S154C, reported ±')
    # This comparison deliberately uses the part label, not an inferred calibrated attenuation.
    ax.plot(group.level, group.nominal_diode_uw*10**(-group.static_part_label_db/10), ':',
            label='Nominal diode × part-label transmission')
    ax.set(title=name, xlabel='Laser command value', ylabel='Power (µW)')
    ax.legend(fontsize=7)
plt.show()


In [ ]:
def parameter_table(values):
    """Flatten settings into a readable, typed table; no JSON or pickled objects."""
    rows = []
    def visit(key, value):
        if is_dataclass(value):
            value = asdict(value)
        if isinstance(value, dict):
            for name, item in value.items():
                visit(f'{key}.{name}' if key else str(name), item)
        elif isinstance(value, (tuple, list, np.ndarray)):
            for i, item in enumerate(value):
                visit(f'{key}.{i}', item)
        else:
            numeric = isinstance(value, (int, float, np.number)) and not isinstance(value, (bool, np.bool_))
            rows.append((key, 'number' if numeric else 'text', float(value) if numeric else np.nan,
                         '' if numeric or value is None else str(value)))
    visit('', values)
    return pd.DataFrame(rows, columns=['key', 'kind', 'number', 'text'])


def parameter(table, key, default=np.nan):
    """Read one saved scalar without reconstructing a driver object."""
    rows = table.loc[table.key.eq(key)]
    if rows.empty:
        return default
    row = rows.iloc[-1]
    return row.number if row.kind == 'number' else row.text if row.text != '' else default


def table_records(frame):
    """Keep numeric dtypes; encode text and stream flag tuples as Unicode for allow_pickle=False."""
    if not len(frame.columns):
        return np.empty(len(frame), dtype=[])
    frame = frame.copy()
    string_types = {}
    for name in frame:
        if frame[name].dtype.kind == 'O' or isinstance(frame[name].dtype, pd.StringDtype):
            frame[name] = frame[name].map(lambda v: '|'.join(v) if isinstance(v, tuple) else '' if v is None else str(v))
            string_types[name] = f'U{max(1, frame[name].str.len().max() if len(frame) else 1)}'
    return frame.to_records(index=False, column_dtypes=string_types)


def save_tables(stem, **tables):
    """Archive raw acquisition before analysis. Exclusive creation never overwrites a capture."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = DATA_DIR / f'{stem}_{stamp}.npz'
    arrays = {name: table_records(value) if isinstance(value, pd.DataFrame) else np.asarray(value)
              for name, value in tables.items()}
    if any(a.dtype.hasobject for a in arrays.values()):
        raise TypeError('NPZ tables must have scalar numeric/Unicode columns, not Python objects.')
    with path.open('xb') as file:
        np.savez_compressed(file, **arrays)
    print(path)
    return path


def load_tables(path):
    """Offline read of this notebook's named arrays and structured tables."""
    with np.load(path, allow_pickle=False) as archive:
        return {key: pd.DataFrame.from_records(archive[key]) if archive[key].dtype.names is not None else archive[key].copy()
                for key in archive.files}


def snapshot(client, laser, *, output=None, fiber=None):
    """Query settings/context; no laser start or PD power-enabling read. Dark/external have no diode identity."""
    output, fiber = output or OUTPUT, fiber or FIBER
    values = dict(utc=datetime.now(timezone.utc).isoformat(), laser=laser, channel=CHANNEL,
        output=output, fiber=fiber, condition=CONDITION, build_label=BUILD_LABEL,
        startup_history=STARTUP_HISTORY, patch_notes=PATCH_NOTES, patch_id=f'{output}_to_{fiber}',
        external_setup=EXTERNAL_SETUP, external_loss_db=EXTERNAL_LOSS_DB, external_loss_basis=EXTERNAL_LOSS_BASIS,
        flashed_adc_sps=FLASHED_ADC_SPS, detector_bandwidth_hz=DETECTOR_BANDWIDTH_HZ,
        adc_input_capacitance_uf=ADC_INPUT_CAPACITANCE_UF,
        status=client.status(), clock=client.time(), heater=client.laser_bankheater(),
        bank=client.laser_bankpower(), pd_settings=client.pd_settings(CHANNEL),
        dark=client.pd_dark(CHANNEL), switches=client.mems(),
        return_loss=client.mems_route_loss(f'{CHANNEL}_{"mm" if fiber == "M" else "sm"}_to_{CHANNEL}_pd'))
    if 'patches' in globals() and values['patch_id'] in patches.index:
        patch = patches.loc[values['patch_id']]
        values.update(external_setup=patch.external_setup, external_loss_db=patch.external_loss_db,
                      external_loss_basis=patch.external_loss_basis)
    if laser in hspcb.LASER_NAMES:
        source = f'{CHANNEL}_1430' if laser.startswith('1430') else f'{CHANNEL}_laser'
        values.update(laser_status=client.laser(laser), engineering=client.laser_status(laser),
            laser_settings=client.laser_settings(laser), coeff=client.atten_coeff(laser), atten=client.atten(laser))
        if output in (f'{CHANNEL}_ao', f'{CHANNEL}_fei'):
            values['launch_loss'] = client.mems_route_loss(f'{source}_to_{output}')
    return parameter_table(values)


def read_route_losses(client, routes):
    """Query current effective losses. The wire value is fraction LOST, not transmission or dB."""
    rows = []
    for route in routes:
        for item in client.mems_route_loss(route).lasers:
            if item.name in LASERS:
                tx = 1.0-float(item.value)
                rows.append(dict(route=route, laser=item.name, transmission=tx,
                    loss_db=-10*np.log10(tx) if tx > 0 else np.nan))
    return pd.DataFrame(rows, columns=['route', 'laser', 'transmission', 'loss_db'])


In [ ]:
def stationary_segments(frame, settle_s=1.0, minimum_s=2.0):
    """Keep separate contiguous fixed-setting intervals; do not close gaps by dropping bad rows."""
    if len(frame) < 3:
        return []
    frame = frame.reset_index(drop=True)
    dt = frame.t_ms.diff()/1000
    nominal = dt[dt > 0].median()
    valid = np.isfinite(frame.pd_net_mv) & np.isfinite(frame.pd_mv) & frame.pd_mv.lt(hspcb.PD_ADC_USABLE_MV)
    valid &= ~frame.autolevel.astype(bool)
    boundary = (dt <= 0) | (dt > 1.5*nominal) | ~valid | ~valid.shift(fill_value=False)
    for column in ('channel', 'laser', 'experiment', 'point', 'repeat', 'step', 'step_index', 'selection'):
        if column in frame:
            labels = frame[column].astype('string').fillna('')
            boundary |= labels.ne(labels.shift(fill_value=''))
    # 0.1 mA steps are the experiment: never merge them using the old 0.2 mA tolerance.
    for column, tolerance in (('laser_current_ma', .049), ('atten_db', .005)):
        boundary |= frame[column].diff().abs().gt(tolerance)
    parts = []
    for _, group in frame.groupby(boundary.cumsum(), sort=False):
        if not valid.loc[group.index].all():
            continue
        group = group.loc[(group.t_ms-group.t_ms.iloc[0])/1000 >= settle_s]
        if len(group) >= 3 and (group.t_ms.iloc[-1]-group.t_ms.iloc[0])/1000 >= minimum_s:
            parts.append(group.reset_index(drop=True))
    return parts


def noise_statistics(part):
    """Empirical scatter, drift, spectrum, ACF and averaging; shared calibration errors are separate."""
    y = part.pd_net_mv.to_numpy(float)
    t = (part.t_ms.to_numpy(float)-float(part.t_ms.iloc[0]))/1000
    dt, rms = np.diff(t), np.std(y, ddof=1)
    fs = 1/np.median(dt)
    centered = y-y.mean()
    acf = signal.correlate(centered, centered, mode='full', method='fft')[len(y)-1:]/np.arange(len(y), 0, -1)
    acf = acf/acf[0] if acf[0] > 0 else np.full_like(acf, np.nan)
    f, psd = signal.welch(y, fs=fs, nperseg=min(1024, max(3, len(y)//4)), detrend='constant')
    blocks = []
    for count in np.unique(np.maximum(1, np.round(np.array([.05,.1,.25,.5,1,2,5,10,20,30,60])*fs).astype(int))):
        n = len(y)//count
        if n < 4:
            continue
        means = y[:n*count].reshape(n, count).mean(axis=1)
        blocks.append(dict(tau_s=count/fs, blocks=n, rms_mean_mv=np.std(means, ddof=1),
            iid_rms_mean_mv=rms/np.sqrt(count), allan_mv=np.sqrt(.5*np.mean(np.diff(means)**2)),
            adjacent_block_corr=np.corrcoef(means[:-1], means[1:])[0,1] if np.std(means)>0 else np.nan))
    metrics = dict(samples=len(y), duration_s=t[-1], mean_mv=y.mean(), rms_mv=rms,
        detrended_rms_mv=np.std(signal.detrend(y), ddof=1), drift_mv_per_s=np.polyfit(t, y, 1)[0],
        relative_rms=rms/abs(y.mean()) if abs(y.mean()) > 3*rms else np.nan,
        adjacent_corr=acf[1], sample_hz=fs, max_gap_ms=1000*max(dt), timing_jitter_ms=1000*np.std(dt))
    return metrics, pd.DataFrame(dict(f_hz=f, psd_mv2_per_hz=psd)), pd.DataFrame(blocks), acf


In [ ]:
def review_capture(path, *, settle_s=1., minimum_s=2., show=True, detailed=False):
    """Offline trace and statistics shared by acquisition and replay. No commands, writes, or calibration changes."""
    saved = load_tables(path)
    data, context = saved.get('samples', pd.DataFrame()), saved.get('context', parameter_table({}))
    if data.empty:
        if show:
            print(f'{Path(path).name}: no samples; inspect the archived outcome/context.')
        return {key: pd.DataFrame() for key in ('audit', 'noise', 'spectra', 'averaging', 'throughput')}
    experiment = str(data.experiment.iloc[0]) if 'experiment' in data else 'capture'
    identity = dict(file=str(path), laser='none' if experiment == 'dark' else parameter(context, 'laser', 'unspecified'), experiment=experiment,
        condition=parameter(context, 'condition', 'unspecified'), build=parameter(context, 'build_label', 'unspecified'),
        patch_id=parameter(context, 'patch_id', 'unspecified'),
        external_setup=parameter(context, 'external_setup', parameter(context, 'patch_notes', 'unspecified')),
        external_loss_db=parameter(context, 'external_loss_db'),
        external_loss_basis=parameter(context, 'external_loss_basis', 'unknown'),
        ambient_c=parameter(context, 'status.amb_c'))
    dt = data.t_ms.diff()/1000
    t = (data.t_ms-data.t_ms.iloc[0])/1000
    usable = np.isfinite(data.pd_mv) & np.isfinite(data.pd_net_mv) & data.pd_mv.lt(hspcb.PD_ADC_USABLE_MV)
    audit = pd.DataFrame([dict(**identity, rows=len(data), duration_s=t.iloc[-1],
        median_dt_ms=1000*dt[dt>0].median(), gaps_over_75ms=int((dt>.075).sum()),
        duplicate_or_reversed=int((dt<=0).sum()), overrange=int((data.pd_mv>=hspcb.PD_ADC_USABLE_MV).sum()),
        autolevel_rows=int(data.autolevel.sum()), raw_mean_mv=data.pd_mv.mean(), raw_rms_mv=data.pd_mv.std(),
        acquisition_complete=parameter(saved.get('outcome', parameter_table({})), 'complete', 'not recorded'))])
    noise_rows, spectra, averaging = [], [], []
    for interval, part in enumerate(stationary_segments(data, settle_s, minimum_s)):
        stats, psd, blocks, acf = noise_statistics(part)
        tags = dict(**identity, interval=interval,
            point=str(part.point.iloc[0]) if 'point' in part else experiment)
        noise_rows.append(dict(**tags, start_t_ms=int(part.t_ms.iloc[0]),
            start_s=(part.t_ms.iloc[0]-data.t_ms.iloc[0])/1000, end_s=(part.t_ms.iloc[-1]-data.t_ms.iloc[0])/1000,
            current_ma=part.laser_current_ma.median(), atten_db=part.atten_db.median(),
            dac1_mv=parameter(context, 'atten.v1_mv'), dac2_mv=parameter(context, 'atten.v2_mv'),
            effective_gain_v_per_a=parameter(context, 'pd_settings.transimpedance_v_per_a'),
            active_dark_rms_mv=parameter(context, 'dark.dark.rms_mv'), **stats))
        spectra.append(psd.assign(**tags))
        averaging.append(blocks.assign(**tags))
    noise = pd.DataFrame(noise_rows)
    valid = data.loc[usable & np.isfinite(data.tp)]
    throughput = pd.DataFrame()
    if not valid.empty:
        recomputed = valid.pd_power_nw/valid.delivered_power_nw.where(valid.delivered_power_nw > 0)
        throughput = pd.DataFrame([dict(**identity, count=len(valid), median_tp=valid.tp.median(),
            p95_tp=valid.tp.quantile(.95), peak_tp=valid.tp.max(), temporal_tp_rms=valid.tp.std(),
            median_model_uncertainty=valid.tp_err.median(), median_pd_uncertainty=valid.tp_pd_err.median(),
            pd_route_tx=valid.pd_route_tx.median(), launch_route_tx=valid.laser_route_tx.median(),
            recompute_max_abs_error=(recomputed-valid.tp).abs().max())])
    if show:
        print(f'{Path(path).name} · {identity["laser"]} · {identity["patch_id"]} · {identity["external_setup"]}')
        display(audit.drop(columns=list(identity)))
        setting_columns = [c for c in ('point', 'selection', 'repeat', 'step', 'requested_current_ma',
            'applied_current_ma', 'measured_current_ma', 'current_match') if c in data]
        if setting_columns:
            display(data[setting_columns].drop_duplicates())
        columns = [] if experiment in ('dark', 'route') else [('laser_current_ma', 'Current set (mA)'), ('atten_db', 'Pair attenuation (dB)')]
        if not throughput.empty:
            columns.append(('tp', 'Throughput'))
        fig, axes = plt.subplots(1+len(columns), 1, sharex=True,
            figsize=(11, 3+1.5*len(columns)), layout='constrained', squeeze=False)
        axes = axes[:, 0]
        axes[0].plot(t, data.pd_mv, '.', ms=2, color='.55', label='Raw PD')
        if experiment != 'dark':
            axes[0].plot(t, data.pd_net_mv, lw=.7, label='Dark-subtracted PD')
        else:
            dark_mean = parameter(context, 'dark.dark.mean_mv')
            axes[0].axhline(dark_mean, color='tab:orange', ls='--', label='Firmware dark mean')
        clipped = data.pd_mv.ge(hspcb.PD_ADC_USABLE_MV)
        if clipped.any():
            axes[0].plot(t[clipped], data.pd_mv[clipped], 'rx', label='Overrange')
        axes[0].set_ylabel('PD (mV)')
        for ax, (column, label) in zip(axes[1:], columns):
            ax.plot(t, data[column], lw=.7)
            ax.set_ylabel(label)
        axes[0].axvspan(0, min(settle_s, t.iloc[-1]), color='.5', alpha=.12, label='Initial analysis exclusion')
        axes[0].legend(fontsize=8)
        axes[-1].set_xlabel('Seconds from first retained sample')
        fig.suptitle(f'{experiment} · {identity["laser"]} · {identity["condition"]} · {identity["build"]}')
        print(f'Statistics exclude {settle_s:g} s after each fixed-setting interval starts; minimum retained interval {minimum_s:g} s.')
        if noise.empty:
            print('No usable stationary interval at these analysis cuts; raw samples remain above and in the NPZ.')
        else:
            display(noise[['interval', 'point', 'start_s', 'end_s', 'samples', 'duration_s', 'mean_mv', 'rms_mv',
                           'drift_mv_per_s', 'relative_rms', 'current_ma', 'atten_db']].rename(columns={'mean_mv': 'mean_net_mv', 'rms_mv': 'rms_net_mv'}))
        if not throughput.empty:
            display(throughput.drop(columns=list(identity)))
        if detailed and 'telemetry' in saved and not saved['telemetry'].empty and 'curr_meas_ma' in saved['telemetry']:
            telemetry = saved['telemetry']
            tt = ((telemetry.host_start_ms+telemetry.host_end_ms)/2-telemetry.host_start_ms.iloc[0])/1000
            tf, ta = plt.subplots(2, 1, sharex=True, figsize=(11, 5), layout='constrained')
            ta[0].plot(tt, telemetry.i_mA, '.-', label='Confirmed setpoint')
            ta[0].plot(tt, telemetry.curr_meas_ma, '.-', label='Measured (0.1 mA resolution)')
            ta[0].set_ylabel('Current (mA)')
            for column in ('tec_temp_c', 'pcb_temp_c', 'ambient_c'):
                ta[1].plot(tt, telemetry[column], '.-', label=column)
            ta[1].set(xlabel='Host seconds from first engineering read', ylabel='Temperature (°C)')
            for ax in ta:
                ax.legend(fontsize=8)
            tf.suptitle(Path(path).name+' · slow engineering telemetry')
        plt.show()
    return dict(audit=audit, noise=noise, throughput=throughput,
        spectra=pd.concat(spectra, ignore_index=True) if spectra else pd.DataFrame(),
        averaging=pd.concat(averaging, ignore_index=True) if averaging else pd.DataFrame())


In [ ]:
if CONNECT:
    if pcb is None:
        pcb = hspcb.HispecFibPcb(BROKER, device=DEVICE, connect=True, auto_connect=True)
    # elif not pcb.is_connected:
    #     pcb.connect()
    inventory = pd.concat([snapshot(pcb, name).assign(source_laser=name) for name in LASERS], ignore_index=True)
    inventory_rows = []
    for name, group in inventory.groupby('source_laser', sort=False):
        inventory_rows.append(dict(laser=name, firmware=parameter(group, 'status.fw'),
            board=parameter(group, 'status.board'), relay_errors=parameter(group, 'status.relay_err'),
            serial=parameter(group, 'engineering.serial'), expected_serial=parameter(group, 'engineering.expected_serial'),
            serial_ok=parameter(group, 'engineering.serial_ok'), ready=parameter(group, 'engineering.ready'),
            current_ma=parameter(group, 'engineering.i_mA'), dark_mv=parameter(group, 'dark.dark.mean_mv')))
    display(pd.DataFrame(inventory_rows))
    inventory_file = save_tables('inventory', inventory=inventory)
else:
    print('Offline: definitions and archived-data analysis are available.')


## Board operation and dark

The selected channel covers three laser/attenuator pairs and four MEMS switches; select HK to commission the
other half. The inventory summary shows identity and communication/readback information; its NPZ retains
full settings. Static MEMS commands save switch intent automatically, but replies do not establish optical routing.

The power exercise checks commands and reported modes. It does not measure temperature change or startup
response. Heater policy persists automatically; finish with the desired mode. Bank `override_off` stops all
lasers and changes thermal history. `pd(channel)` powers/refreshes the detector, so the power check instead
queries `pd_settings` for mode readback. That API does not expose selected-relay power confirmation.

For dark, ensure external/manual inputs are also dark. Inspect the plotted validation trace before accepting
its preceding firmware dark window. They have different durations; transient startup can make them disagree.
Acceptance saves that measured window and resets the lowest-dark record to it, preserving measured RMS and
dark-mean uncertainty. No fixed warmup time or automatic acceptance threshold is assumed here.


In [ ]:
RUN_POWER_EXERCISE = True
POWER_STEPS = [('bank', 'override_on'), ('heater', 'override_on'), ('heater', 'override_off'),
               ('heater', 'auto'), ('pd', 'override_off'), ('pd', 'override_on'),
               ('pd', 'auto'), ('bank', 'override_off'), ('bank', 'auto')]
if RUN_POWER_EXERCISE:
    pcb.stop_throughput('all')
    power_rows = []
    try:
        for component, mode in POWER_STEPS:
            row = dict(component=component, requested=mode, reported='', powered=None, error='')
            try:
                if component == 'bank':
                    pcb.laser_bankpower(mode)
                    state = pcb.laser_bankpower()
                    row.update(reported=state.mode, powered=state.powered)
                elif component == 'heater':
                    pcb.laser_bankheater(mode)
                    state = pcb.laser_bankheater()
                    row.update(reported=state.mode, powered=state.heater_on, error=str(state.last_error) if state.last_error else '')
                else:
                    pcb.pd_settings(CHANNEL, power=mode, persist=False)
                    row['reported'] = pcb.pd_settings(CHANNEL).power
            except Exception as exc:
                row['error'] = str(exc)
                raise
            finally:
                power_rows.append(row)
    finally:
        display(pd.DataFrame(power_rows))  # Command/readback check: no scientific archive needed.


In [ ]:
def open_stream(client, seconds, *, laser='none', fiber=None, output=None, autolevel=False, initial_level=None):
    """Stop an earlier owner BEFORE manual source setup; return a new finite-capacity collector.

    A former autolevel owner can shut down its source here. The returned manual/passive
    collector has no inherited shutdown obligation. Its stop leaves manually enabled lasers alone.
    """
    if not np.isfinite(seconds) or seconds <= 0:
        raise ValueError('Choose a positive capture duration.')
    client.stop_throughput(CHANNEL)
    return client.measure_throughput(laser, channel=CHANNEL, fiber=fiber or FIBER,
        output=(output or OUTPUT) if laser != 'none' else None, autolevel=autolevel,
        initial_level=initial_level, off_in_s=0, collect=True, format='binary',
        max_samples=max(1000, math.ceil(seconds / .05 * 1.5) + 1000))


def collect_trace(client, monitor, laser, seconds, label, *, context, extra=None, telemetry_s=2.0):
    """Block for an editable hold; save on completion or interrupt and detach the stream.

    Slow engineering queries give measured current/TEC context, not high-frequency current noise.
    Raw samples include the start transient; settling cuts belong in analysis. No automatic laser STOP.
    """
    started = time.monotonic()
    readings, error, completed = [], '', False
    try:
        next_read = started
        while time.monotonic() - started < seconds:
            if time.monotonic() >= next_read:
                before = time.time_ns() // 1_000_000
                status = client.laser_status(laser) if laser in hspcb.LASER_NAMES else None
                ambient = client.status().amb_c
                heater = client.laser_bankheater()
                after = time.time_ns() // 1_000_000
                # PID configuration is already in the context table; each row keeps scalar telemetry.
                values = {k: v for k, v in asdict(status).items() if k != 'pid'} if status is not None else {}
                readings.append(dict(host_start_ms=before, host_end_ms=after, ambient_c=ambient,
                    heater_on=heater.heater_on, heater_auto_state=heater.auto_state, **values))
                next_read = time.monotonic() + telemetry_s
            time.sleep(min(.1, max(0, seconds - (time.monotonic() - started))))
        completed = True
    except BaseException as exc:
        error = f'{type(exc).__name__}: {exc}'
        raise
    finally:
        # Stop/detach first to include queued telemetry. Save even if the stop command fails.
        stop_error = ''
        try:
            monitor.stop()
        except Exception as exc:
            stop_error = f'{type(exc).__name__}: {exc}'
            raise
        finally:
            frame = monitor.to_dataframe()
            for name, value in dict(experiment=label, source_laser=laser, condition=CONDITION,
                                    build_label=BUILD_LABEL, **(extra or {})).items():
                frame[name] = value
            telemetry = pd.DataFrame(readings)
            outcome = parameter_table(dict(complete=completed, error=error, stop_error=stop_error, requested_s=seconds,
                elapsed_s=time.monotonic()-started, samples=len(frame), collector_capacity=monitor.max_samples,
                capacity_reached=len(frame) >= monitor.max_samples))
            path = save_tables(f'{label}_{laser}', samples=frame, telemetry=telemetry, context=context, outcome=outcome,
                               source_reference=source_power_readings, nominal_reference=laser_nominal.reset_index())
            CAPTURE_FILES.append(path)
            if label == 'route':
                ROUTE_FILES.append(path)
            # Archive first: a display failure must not lose acquisition data or hide an earlier error.
            try:
                review_capture(path)
            except Exception as display_error:
                print(f'Capture saved; offline review failed: {display_error}')
    return path


In [ ]:
RUN_DARK = True
DARK_DURATION_MS = 2000
DARK_TRACE_SECONDS = 20.0
if RUN_DARK:
    pcb.stop_throughput('all')
    for name in LASERS:
        pcb.laser(name, value=0, autooff_s=0)
    # Confirm all external/manual inputs to this PD are dark as well.
    pcb.pd(CHANNEL)
    dark = pcb.pd_dark(CHANNEL, duration_ms=DARK_DURATION_MS, persist=False)
    while dark.pending:
        time.sleep(.25)
        dark = pcb.pd_dark(CHANNEL)
    reviewed_dark = dark.dark
    display(pd.DataFrame([asdict(reviewed_dark)]).assign(measurement='firmware dark window'))
    dark_file = save_tables('dark_settings', dark=parameter_table(asdict(dark)))
    context = snapshot(pcb, 'none')
    monitor = open_stream(pcb, DARK_TRACE_SECONDS)
    dark_trace_file = collect_trace(pcb, monitor, 'none', DARK_TRACE_SECONDS, 'dark', context=context)


In [ ]:
ACCEPT_DARK = True                    # Run separately after reviewing the window and validation trace.
if ACCEPT_DARK:
    active = pcb.pd_dark(CHANNEL)
    dark_fields = ('duration_ms', 'failed_samples', 'mean_mv', 'rms_mv', 'min_mv', 'max_mv')
    if active.pending or reviewed_dark.duration_ms <= 0 or not np.isfinite([reviewed_dark.mean_mv, reviewed_dark.rms_mv]).all() or not np.allclose(
            [getattr(active.dark, key) for key in dark_fields],
            [getattr(reviewed_dark, key) for key in dark_fields], rtol=0, atol=0, equal_nan=True):
        raise RuntimeError('Active dark differs from the reviewed measured window; capture and review again.')
    pcb.pd_dark(CHANNEL, reset_lowest=True, persist=True)
    accepted_dark = pcb.pd_dark(CHANNEL)
    display(pd.DataFrame([asdict(accepted_dark.dark), asdict(accepted_dark.lowest_dark)],
                         index=['saved active dark', 'reset lowest dark']))


## Physical routes and connector identity

A patch is a physical output connected through your external setup to one return fiber. Laser/CAL selection
is a separate setting; switching sources does not change the patch ID. The physical table includes AO, FEI,
and all four 1430 retro splitter outputs, each to M or S. Edit its setup description and loss knowledge as the
bench pad is replaced by an instrument. An unknown transmission does not prevent a routing measurement.

Select one patch and one source below, run its intended → alternate return → alternate launch → restored
sequence, then select another source without repatching. Change the physical connection only between groups.
The plots retain initial samples; the later comparison applies an explicit settling cut. Neither a successful
command nor a dark alternate route alone establishes that every connector is correct.

For retro, `forward_retro` is driven directly to B; the splitter is passive. Return-only `laser='none'`
streaming leaves launch selection alone. External CAL light is supplied manually, with bank lasers zeroed.
Set visible, unclipped attenuation before acquiring: the initial 3300/3300 mV values below are parked placeholders.


In [ ]:
patches = pd.DataFrame([
    dict(patch_id=f'{CHANNEL}_{endpoint}_to_{fiber}', endpoint=endpoint, fiber=fiber,
         external_setup=EXTERNAL_SETUP, external_loss_db=EXTERNAL_LOSS_DB, external_loss_basis=EXTERNAL_LOSS_BASIS)
    for endpoint in ('ao', 'fei', 'retro_1', 'retro_2', 'retro_3', 'retro_4')
    for fiber in ('M', 'S')]).set_index('patch_id')
# Edit the selected physical path here; repeated source tests use that same row.
# patches.loc['yj_ao_to_M', ['external_setup', 'external_loss_db', 'external_loss_basis']] = ['bench pad ...', 3., 'estimated']
display(patches)


In [ ]:
route_rows = []
for patch_id, patch in patches.iterrows():
    sources = [f'1430{CHANNEL}'] if patch.endpoint.startswith('retro_') else [*LASERS, 'external']
    for name in sources:
        kind = 'retro' if patch.endpoint.startswith('retro_') else 'cal' if name == 'external' else 'forward'
        source = f'{CHANNEL}_cal' if name == 'external' else f'{CHANNEL}_1430' if name.startswith('1430') else f'{CHANNEL}_laser'
        route_rows.append(dict(patch_id=patch_id, laser=name, source=source, kind=kind,
            level=0. if name == 'external' else .1, dac1_mv=3300., dac2_mv=3300., duration_s=5., repeats=2))
route_plan = pd.DataFrame(route_rows).set_index(['patch_id', 'laser'])
PATCH_ID, ROUTE_LASER = f'{CHANNEL}_ao_to_M', LASERS[0]
# The meter references are at .1, .5 and 1.; other levels still give raw route measurements.
# Enter proven visible-light settings; 3300/3300 is only the initial parked state.
# route_plan.loc[(PATCH_ID, ROUTE_LASER), ['dac1_mv', 'dac2_mv']] = [2500., 2500.]
display(route_plan.loc[PATCH_ID])


In [ ]:
RUN_ROUTE_GROUP = False
if RUN_ROUTE_GROUP:
    patch, row = patches.loc[PATCH_ID], route_plan.loc[(PATCH_ID, ROUTE_LASER)]
    display(pd.DataFrame([{**patch.to_dict(), **row.to_dict(), 'patch_id': PATCH_ID, 'laser': ROUTE_LASER}]))
    states = [('intended', patch.fiber, False), ('alternate_return', 'S' if patch.fiber == 'M' else 'M', False),
              ('alternate_launch', patch.fiber, True), ('restored', patch.fiber, False)]
    for repeat in range(int(row.repeats)):
        for label, selected_fiber, alternate in states:
            # Stop the former owner before source setup. Passive collection changes only the return selection.
            monitor = open_stream(pcb, row.duration_s, fiber=selected_fiber)
            try:
                for name in LASERS:
                    pcb.laser(name, value=0, autooff_s=0)
                if row.kind == 'retro':
                    endpoint = patch.endpoint
                    pcb.mems_switch(f'{CHANNEL}_forward_retro', state='A' if alternate else 'B')
                else:
                    endpoint = ('fei' if patch.endpoint == 'ao' else 'ao') if alternate else patch.endpoint
                    pcb.mems_route(row.source, f'{CHANNEL}_{endpoint}')
                if ROUTE_LASER != 'external':
                    pcb.atten(ROUTE_LASER, value1_mv=row.dac1_mv, value2_mv=row.dac2_mv)
                    pcb.laser(ROUTE_LASER, value=row.level, autooff_s=0)
                context = pd.concat([snapshot(pcb, ROUTE_LASER, fiber=selected_fiber, output=f'{CHANNEL}_{endpoint}'),
                    parameter_table(dict(patch_id=PATCH_ID, patch=patch.to_dict(), route_test=row.to_dict(),
                        external_setup=patch.external_setup, external_loss_db=patch.external_loss_db,
                        external_loss_basis=patch.external_loss_basis))], ignore_index=True)
                collect_trace(pcb, monitor, ROUTE_LASER, row.duration_s, 'route', context=context,
                    extra=dict(patch_id=PATCH_ID, selection=label, repeat=repeat, selected_fiber=selected_fiber,
                               source_level=row.level))
            finally:
                try:
                    monitor.stop()
                finally:
                    if ROUTE_LASER != 'external':
                        pcb.laser(ROUTE_LASER, value=0, autooff_s=0)


## Assembled static losses

Start with the **current firmware settings**, then enter installed switch serials and the directional,
wavelength-specific values you select manually from the MEMS workbook. There is no automatic spreadsheet
reader. Optical ports **a/b/c** and control states **A/B** are different concepts: fill the optical direction
from your lab record. Component sums are rough comparisons; optical measurements are required.

The component worksheet below names complete firmware paths. Static attenuator part labels are only nominal
entries, FVOA open insertion loss and connectors initially remain unknown, and dynamic attenuation is separate.
Known subtotals appear immediately; a complete total stays missing until every component has a value.

Route captures provide signed detector power from **net voltage / (responsivity × effective gain)**, without
applying firmware return transmission a second time. The meter-relative ratio uses the supplied S154C sensor
plane, so its unknown measurement-cable loss remains in the comparison. The diode-model ratio instead uses the
queried current/efficiency model. Neither ratio automatically identifies individual PCB route losses.

Inspect raw traces and edit corrections/reference planes in the measurement worksheet before using a result.
A known or estimated external loss can be subtracted explicitly. Unknown instrument loss stays unknown; combined
loopback loss is never automatically assigned between launch and return. Only the separate manual update table
writes route calibration, with distinct RAM and persistence cells and a visible readback.


In [ ]:
TIB_ROUTES = [f'{CHANNEL}_{source}_to_{CHANNEL}_{output}'
    for source in ('1430', 'cal', 'laser') for output in ('ao', 'fei')]
TIB_ROUTES += [f'{CHANNEL}_{fiber}_to_{CHANNEL}_pd' for fiber in ('mm', 'sm')]
QUERY_ROUTE_LOSSES = CONNECT
route_loss_settings = globals().get('route_loss_settings', pd.DataFrame(columns=['route', 'laser', 'transmission', 'loss_db']))
if QUERY_ROUTE_LOSSES:
    route_loss_settings = read_route_losses(pcb, TIB_ROUTES)
if route_loss_settings.empty:
    print('No queried route settings yet. Run this query when connected; offline references remain editable below.')
else:
    print('Current effective firmware loss (dB); zero may be a default, not a measured loss:')
    display(route_loss_settings.pivot(index='route', columns='laser', values='loss_db'))
    print('Corresponding transmission:')
    display(route_loss_settings.pivot(index='route', columns='laser', values='transmission'))


In [ ]:
REFERENCE_LASER = LASERS[0]             # Limits the displayed worksheet; all channel paths are retained.
installed_switches = pd.DataFrame([
    (f'{CHANNEL}_forward_retro', 'sw1' if CHANNEL == 'yj' else 'sw4', 'SW1_TIBB1' if CHANNEL == 'yj' else 'SW1_TIBR1', ''),
    (f'{CHANNEL}_laser_cal', 'sw2' if CHANNEL == 'yj' else 'sw5', 'SW1_TIBB2' if CHANNEL == 'yj' else 'SW1_TIBR2', ''),
    (f'{CHANNEL}_ao_fei', 'sw3' if CHANNEL == 'yj' else 'sw6', 'SW1_TIBB3' if CHANNEL == 'yj' else 'SW1_TIBR3', ''),
    (f'{CHANNEL}_mm_sm', 'sw7' if CHANNEL == 'yj' else 'sw8', 'SW2_FFLS1' if CHANNEL == 'yj' else 'SW2_FFLS2', '')],
    columns=['location', 'pcb_switch', 'hardware_name', 'serial']).set_index('location')
switch_losses = pd.DataFrame([
    dict(reading_id=f'{CHANNEL}_{location}:{branch}:{name}', location=f'{CHANNEL}_{location}', laser=name,
         branch=branch, optical_direction='', wavelength_nm=laser_nominal.loc[name, 'wavelength_nm'], loss_db=np.nan)
    for name in LASERS
    for location, branches in [('forward_retro', ('forward', 'retro')), ('laser_cal', ('laser', 'cal')),
                               ('ao_fei', ('ao', 'fei')), ('mm_sm', ('M', 'S'))]
    for branch in branches]).set_index('reading_id')
# Enter serials and selected spreadsheet measurements explicitly; e.g. edit these assignments:
# installed_switches.loc['yj_ao_fei', 'serial'] = 'measured unit serial'
# switch_losses.loc['yj_ao_fei:ao:1028y', ['optical_direction', 'loss_db']] = ['a->b', YOUR_MEASURED_DB]
display(installed_switches, switch_losses.loc[switch_losses.laser.eq(REFERENCE_LASER)])

component_rows = []
for name in LASERS:
    source = '1430' if name.startswith('1430') else 'laser'
    part = source_power_readings.loc[source_power_readings.laser.eq(name), 'static_part_label_db']
    static = float(part.iloc[0]) if len(part) else np.nan
    for endpoint in ('ao', 'fei'):
        for origin in (source, 'cal'):
            parts = [] if origin == 'cal' else [
                ('static attenuator', '', static, 'part label only'), ('FVOA pair open insertion', '', np.nan, 'unknown')]
            if origin == '1430':
                parts += [('forward selector', f'{CHANNEL}_forward_retro:forward:{name}', np.nan, 'switch measurement')]
            parts += [('laser/CAL selector', f'{CHANNEL}_laser_cal:{"cal" if origin == "cal" else "laser"}:{name}', np.nan, 'switch measurement'),
                      ('output selector', f'{CHANNEL}_ao_fei:{endpoint}:{name}', np.nan, 'switch measurement'),
                      ('other internal connectors/fiber', '', np.nan, 'unknown')]
            for component, reading, loss, basis in parts:
                component_rows.append(dict(path_id=f'{CHANNEL}_{origin}_to_{CHANNEL}_{endpoint}', laser=name,
                    component=component, switch_reading=reading, loss_db=loss, basis=basis))
    for fiber, word in [('M', 'mm'), ('S', 'sm')]:
        for component, reading in [('return selector', f'{CHANNEL}_mm_sm:{fiber}:{name}'), ('return connectors/fiber', '')]:
            component_rows.append(dict(path_id=f'{CHANNEL}_{word}_to_{CHANNEL}_pd', laser=name,
                component=component, switch_reading=reading, loss_db=np.nan, basis='switch measurement' if reading else 'unknown'))
    if source == '1430':
        for port in range(1, 5):
            for component, reading, loss, basis in [
                ('static attenuator', '', static, 'part label only'), ('FVOA pair open insertion', '', np.nan, 'unknown'),
                ('retro selector', f'{CHANNEL}_forward_retro:retro:{name}', np.nan, 'switch measurement'),
                ('splitter port and connectors', '', np.nan, 'unknown')]:
                component_rows.append(dict(path_id=f'{CHANNEL}_retro_{port}', laser=name, component=component,
                    switch_reading=reading, loss_db=loss, basis=basis))
path_components = pd.DataFrame(component_rows)
# Edit direct loss_db/basis entries for connectors, open insertion, and measured static attenuators.
# Switch rows obtain loss_db from switch_losses in the next cell; change their reading ID to select another measurement.
display(path_components.loc[path_components.laser.eq(REFERENCE_LASER)])


In [ ]:
component_reference = path_components.copy()
use_switch = component_reference.switch_reading.ne('')
component_reference.loc[use_switch, 'loss_db'] = component_reference.loc[use_switch, 'switch_reading'].map(switch_losses.loss_db)
component_reference['serial'] = component_reference.switch_reading.map(switch_losses.location).map(installed_switches.serial)
component_reference['optical_direction'] = component_reference.switch_reading.map(switch_losses.optical_direction)
component_reference['wavelength_nm'] = component_reference.switch_reading.map(switch_losses.wavelength_nm)
path_totals = component_reference.groupby(['path_id', 'laser'], sort=False).loss_db.agg(
    known_subtotal_db=lambda s: s.sum(min_count=1), unknown_components=lambda s: int(s.isna().sum()),
    reference_db=lambda s: s.sum(min_count=len(s))).reset_index()
print('Rough component references; nominal part labels are not calibrated losses:')
display(path_totals.loc[path_totals.laser.eq(REFERENCE_LASER)])


In [ ]:
# ROUTE_FILES is filled by route acquisition, including partial captures. Add old paths explicitly for replay.
ROUTE_SETTLE_S = 1.0                   # Analysis cut only; initial samples remain in NPZ and the plots.
route_summary_rows = []
for path in dict.fromkeys(ROUTE_FILES):
    saved = load_tables(path)
    data, ctx = saved.get('samples', pd.DataFrame()), saved['context']
    if data.empty:
        print(f'{Path(path).name}: no route samples.')
        continue
    after_settle = (data.t_ms-data.t_ms.iloc[0])/1000 >= ROUTE_SETTLE_S
    f = data.loc[after_settle]
    usable = f.loc[np.isfinite(f.pd_net_mv) & np.isfinite(f.pd_mv) & f.pd_mv.lt(hspcb.PD_ADC_USABLE_MV)]
    name = parameter(ctx, 'laser', 'external')
    level = parameter(ctx, 'route_test.level')
    r, g = parameter(ctx, 'pd_settings.responsivity_a_per_w'), parameter(ctx, 'pd_settings.transimpedance_v_per_a')
    detector_nw = usable.pd_net_mv.mean()*1e6/(r*g) if np.isfinite(r*g) and r*g > 0 else np.nan
    reference = saved.get('source_reference', pd.DataFrame(columns=source_power_readings.columns))
    matched = reference.loc[reference.laser.eq(name) & np.isclose(reference.level.astype(float), level, rtol=0, atol=1e-12)]
    meter_nw = float(matched.power_uw.iloc[0])*1000 if len(matched) == 1 else np.nan
    current, threshold = parameter(ctx, 'engineering.i_mA'), parameter(ctx, 'laser_settings.threshold_current_ma')
    diode_nw = max(0., current-threshold)*parameter(ctx, 'laser_settings.efficiency_mw_per_ma')*1e6 if np.isfinite(current+threshold) else np.nan
    launch_path = parameter(ctx, 'launch_loss.route', parameter(ctx, 'output', ''))
    if parameter(ctx, 'route_test.kind', '') == 'retro':
        launch_path = parameter(ctx, 'output', '')
    return_path = parameter(ctx, 'return_loss.route', '')
    firmware_losses = []
    for key in ('launch_loss', 'return_loss'):
        ids = ctx.loc[ctx.key.str.startswith(key+'.lasers.') & ctx.key.str.endswith('.name')]
        matches = ids.loc[ids.text.eq(name), 'key']
        lost = parameter(ctx, matches.iloc[0][:-4]+'value') if len(matches) else np.nan
        firmware_losses.append(-10*np.log10(1-lost) if np.isfinite(lost) and lost < 1 else np.nan)
    refs = path_totals.set_index(['path_id', 'laser']).reference_db
    total_ref = refs.get((launch_path, name), np.nan)+refs.get((return_path, name), np.nan)
    route_summary_rows.append(dict(file=str(path), patch_id=parameter(ctx, 'patch_id'), laser=name, level=level,
        selection=data.selection.iloc[0], repeat=data.repeat.iloc[0], count=len(usable),
        external_setup=parameter(ctx, 'external_setup', parameter(ctx, 'patch_notes', 'unspecified')),
        external_loss_db=parameter(ctx, 'external_loss_db'), external_loss_basis=parameter(ctx, 'external_loss_basis', 'unknown'),
        mean_mv=usable.pd_net_mv.mean(), rms_mv=usable.pd_net_mv.std(), overrange=int(f.pd_mv.ge(hspcb.PD_ADC_USABLE_MV).sum()),
        detector_power_nw=detector_nw, meter_reference_nw=meter_nw,
        meter_reported_pm_nw=float(matched.reported_pm_uw.iloc[0])*1000 if len(matched) == 1 else np.nan,
        nominal_diode_nw=diode_nw, launch_path=launch_path, return_path=return_path,
        model_dynamic_db=parameter(ctx, 'atten.db'), firmware_static_db=sum(firmware_losses),
        component_reference_db=total_ref,
        meter_relative_db=10*np.log10(meter_nw/detector_nw) if meter_nw > 0 and detector_nw > 0 else np.nan,
        diode_model_total_db=10*np.log10(diode_nw/detector_nw) if diode_nw > 0 and detector_nw > 0 else np.nan))
route_summary = pd.DataFrame(route_summary_rows)
if route_summary.empty:
    print('No route measurements yet. Run a selected patch/source group; the component totals above are references only.')
else:
    display(route_summary[['patch_id', 'laser', 'level', 'selection', 'repeat', 'count', 'mean_mv', 'rms_mv', 'overrange']])
    matrix = route_summary.groupby(['patch_id', 'laser', 'level', 'external_setup', 'external_loss_db',
        'external_loss_basis', 'repeat', 'selection'], dropna=False).mean_mv.mean().unstack('selection')
    comparison = matrix.reset_index()
    display(comparison)
    plot_values = matrix.copy()
    plot_values.index = [f'{i}: {row.patch_id} / {row.laser} / level {row.level:g}'
                         for i, row in comparison.iterrows()]
    ax = plot_values.plot.barh(xlabel='Signed PD net (mV)', ylabel='Comparison row',
                              figsize=(11, max(3., .5*len(plot_values))))
    ax.figure.set_layout_engine('constrained')
    plt.show()
    print('Ratios retain their distinct reference planes; no loss has been assigned or applied:')
    display(route_summary[['patch_id', 'laser', 'selection', 'detector_power_nw', 'meter_reference_nw',
        'meter_reported_pm_nw', 'meter_relative_db', 'diode_model_total_db', 'model_dynamic_db', 'external_loss_db', 'external_loss_basis',
        'component_reference_db', 'firmware_static_db']])


In [ ]:
# Editable analysis worksheet. Running this cell seeds rows; it does not take measurements or write settings.
loss_measurements = pd.DataFrame(columns=['file', 'patch_id', 'laser', 'reference_plane', 'before_nw', 'after_nw',
    'external_loss_db', 'external_loss_basis', 'dynamic_correction_db', 'other_correction_db', 'notes'])
if not route_summary.empty:
    measured = route_summary.loc[route_summary.selection.isin(['intended', 'restored'])].copy()
    loss_measurements = measured[['file', 'patch_id', 'laser', 'external_loss_db', 'external_loss_basis']].copy()
    loss_measurements['reference_plane'] = 'S154C sensor after static attenuator + unknown measurement fiber'
    loss_measurements['before_nw'] = measured.meter_reference_nw
    loss_measurements['after_nw'] = measured.detector_power_nw
    loss_measurements['dynamic_correction_db'] = np.nan
    loss_measurements['other_correction_db'] = np.nan
    loss_measurements['notes'] = 'Meter-relative ratio; unknown measurement-cable loss remains. See route_summary for diode-model comparison.'
# Positive corrections are subtracted. A loss in the meter's reference arm has the opposite sign; it is unknown here.
# Enter corrections deliberately; use 0 only when a correction does not apply. Do not copy missing values to zero.
# Enter another measured before-plane/value explicitly for external CAL or a different source reference.
display(loss_measurements)


In [ ]:
loss_results = loss_measurements.copy()
good = (loss_results.before_nw > 0) & (loss_results.after_nw > 0)
loss_results['observed_db'] = np.nan
loss_results.loc[good, 'observed_db'] = 10*np.log10(loss_results.loc[good, 'before_nw']/loss_results.loc[good, 'after_nw'])
corrections = loss_results[['external_loss_db', 'dynamic_correction_db', 'other_correction_db']]
loss_results['corrected_db'] = loss_results.observed_db-corrections.sum(axis=1, min_count=3)
loss_results['unresolved_corrections'] = corrections.isna().sum(axis=1)
if loss_results.empty:
    print('No optical measurements in this worksheet yet; nothing was measured, saved, or applied by these analysis cells.')
else:
    display(loss_results)
    print('Corrected ratios keep the stated reference plane and its uncertainties; they do not automatically become individual route losses.')


In [ ]:
# Enter reviewed complete route totals, with supporting evidence/reference planes; no automatic allocation.
route_updates = pd.DataFrame(columns=['route', 'laser', 'loss_db', 'evidence'])
if route_updates.empty:
    print('No route updates entered; no settings will be written.')
else:
    before_updates = read_route_losses(pcb, route_updates.route.unique()) if CONNECT else route_loss_settings
    preview = route_updates.merge(before_updates[['route', 'laser', 'loss_db']].rename(columns={'loss_db': 'current_db'}),
                                  on=['route', 'laser'], how='left', validate='one_to_one')
    display(preview.rename(columns={'loss_db': 'requested_total_db'}))
# Numeric wire values mean fraction LOST. Use dB strings for large losses; relative FVOA fits exclude static loss.


def apply_route_loss_updates(client, updates, *, persist):
    """Apply explicitly entered complete losses and archive acknowledged changes/readback, including partial progress."""
    if updates.empty:
        print('No route updates entered; nothing written or archived.')
        return
    if updates.duplicated(['route', 'laser']).any() or not updates.route.isin(TIB_ROUTES).all() or not updates.laser.isin(LASERS).all():
        raise ValueError('Use unique route/laser pairs from the displayed channel settings.')
    if not (np.isfinite(updates.loss_db) & updates.loss_db.ge(0)).all():
        raise ValueError('Supply finite nonnegative complete route losses in dB.')
    before = read_route_losses(client, updates.route.unique()).set_index(['route', 'laser'])
    client.stop_throughput(CHANNEL)     # Stream route factors are latched; restart after calibration changes.
    changes = []
    try:
        for row in updates.itertuples(index=False):
            client.mems_route_loss(row.route, laser=row.laser, loss=f'{row.loss_db:.12g} dB', persist=persist)
            change = dict(route=row.route, laser=row.laser, before_db=before.loc[(row.route, row.laser), 'loss_db'],
                requested_db=row.loss_db, readback_db=np.nan, persist_requested=persist, evidence=row.evidence)
            changes.append(change)
            after = read_route_losses(client, [row.route]).set_index(['route', 'laser'])
            change['readback_db'] = after.loc[(row.route, row.laser), 'loss_db']
    finally:
        if changes:
            display(pd.DataFrame(changes))
            save_tables('route_loss_update', changes=pd.DataFrame(changes), installed_switches=installed_switches.reset_index(),
                switches=switch_losses.reset_index(), components=component_reference, measurements=loss_results,
                source_reference=source_power_readings,
                context=parameter_table(dict(channel=CHANNEL, external_setup=EXTERNAL_SETUP,
                    utc=datetime.now(timezone.utc).isoformat())))
    print('Persistence was requested and acknowledged.' if persist else 'Applied in RAM only.')
    print('Readback reports current effective values; persistence across a reboot is not tested here.')


In [ ]:
APPLY_ROUTE_LOSSES_RAM = False
if APPLY_ROUTE_LOSSES_RAM:
    apply_route_loss_updates(pcb, route_updates, persist=False)


In [ ]:
PERSIST_ROUTE_LOSSES = False
if PERSIST_ROUTE_LOSSES:
    apply_route_loss_updates(pcb, route_updates, persist=True)


## Embedded autocalibration

Patch the configured output to the selected return and obtain a valid dark first. Each run exercises both physical
FVOAs over the DAC range; firmware may use full nominal laser output. It owns acquisition, bridging, fitting,
and installation of accepted coefficients in RAM. `persist=False` does **not** prevent the RAM update.

Run all three lasers or select a subset below. Each pair is fetched, archived, and plotted before starting another.
Completed acquisitions with rejected fits are still scientific data. Interrupts attempt to fetch the currently
retained data before cancellation; that partial snapshot can be incomplete. Firmware cancellation clears its
records. A retrieval error is recorded and re-raised, never presented as recovered data.

In [ ]:
def save_calibration(dataset, context, laser, *, error=''):
    """Store the firmware result and raw anchors as tables before any cleanup can discard them."""
    fits = next((item['fits'] for item in dataset.meta if 'fits' in item), {})
    fit_rows, physical_rows, bridge_rows = [], [], []
    for physical in ('dac1', 'dac2'):
        fit = fits.get(physical, hspcb.AttenuatorFitMetrics(valid=False))
        row = {k: np.nan if v is None else v for k, v in asdict(fit).items() if k != 'correction_coeff'}
        row.update(physical=physical, **{f'correction_{i}': x for i, x in enumerate(fit.correction_coeff or (0.,)*6)})
        fit_rows.append(row)
        meta = dataset._physical_meta(physical)
        if meta is not None:
            physical_rows.append({k: v for k, v in meta.items() if k != 'bridges'})
            bridge_rows.extend(dict(physical=physical, **bridge) for bridge in meta.get('bridges', ()))
    status = next((item['status'] for item in dataset.meta if 'status' in item), {})
    error = error or next((item.get('retrieval_error', '') for item in dataset.meta if 'retrieval_error' in item), '')
    return save_tables(f'cal_{laser}', records=dataset.records, fits=pd.DataFrame(fit_rows),
        physical=pd.DataFrame(physical_rows, columns=list(physical_rows[0]) if physical_rows else ['physical']),
        bridges=pd.DataFrame(bridge_rows, columns=['physical', 'bridge_index', 'before_record', 'after_record']),
        status=parameter_table({'status': status, 'retrieval_error': error}), context=context,
        source_reference=source_power_readings, nominal_reference=laser_nominal.reset_index())


def load_calibration(path):
    """Read new table archives or the actual older NPZ captures; no hardware connection."""
    with np.load(path, allow_pickle=False) as z:
        records = z['records'].copy()
        if 'metadata' in z.files:
            import json                   # Read-only support for existing Copy1 captures.
            saved = json.loads(str(z['metadata']))
            for item in saved['meta']:
                if 'fits' in item:
                    item['fits'] = {name: hspcb.AttenuatorFitMetrics(**value) for name, value in item['fits'].items()}
            return hspcb.AttenuatorCalibrationDataset(records, tuple(saved['meta'])), parameter_table(saved['context'])
        tables = {key: pd.DataFrame.from_records(z[key]) for key in ('fits', 'physical', 'bridges', 'context', 'status')}
    fits = {}
    for row in tables['fits'].to_dict('records'):
        physical = row['physical']
        values = {f.name: row[f.name] for f in fields(hspcb.AttenuatorFitMetrics) if f.name != 'correction_coeff'}
        values = {k: None if isinstance(v, float) and np.isnan(v) else v for k, v in values.items()}
        values['correction_coeff'] = tuple(row[f'correction_{i}'] for i in range(6))
        fits[physical] = hspcb.AttenuatorFitMetrics(**values)
    status = {f.name: parameter(tables['status'], f'status.{f.name}', None)
              for f in fields(hspcb.AttenuatorCalibrationStatus) if f.name not in ('dac1', 'dac2')}
    if status['state'] is not None:
        for key in ('n', 't_ms', 'complete_pct', 'error'):
            status[key] = int(status[key])
        status = hspcb.AttenuatorCalibrationStatus(**status, dac1=fits['dac1'], dac2=fits['dac2'])
    else:
        status = {}
    meta = [{'fits': fits, 'status': status, 'retrieval_error': parameter(tables['status'], 'retrieval_error', '')}]
    for row in tables['physical'].to_dict('records'):
        row['bridges'] = tables['bridges'].loc[tables['bridges'].physical.eq(row['physical'])].drop(columns='physical').to_dict('records')
        meta.append(row)
    return hspcb.AttenuatorCalibrationDataset(records, tuple(meta)), tables['context']


def captured_coefficients(dataset, context):
    """Use each captured fit with its acquisition gain, never the presently installed coefficients."""
    result = {}
    for physical in ('dac1', 'dac2'):
        coeff = dataset._fit_coeff_for_physical(physical)
        if coeff is not None:
            gain = parameter(context, f'coeff.{physical}.gain', hspcb.ATTENUATOR_DEFAULT_GAIN)
            result[physical] = (*coeff[:3], float(gain), coeff[4], coeff[5])
    return result


## Combined calibration figure and offline replay

The upper panels preserve raw dark-subtracted and bridge-scaled data. The fit overlays measured sweep records,
with a quarter-height residual panel attached directly below it. Faint separators identify segments; the
vertical line labels the DAC voltage crossing into the rough region. The tail shading starts at excluded tail
measurements, separately from the calibrated-limit line.

**Excluded** marks known omissions, including positive low-SNR points. Valid above-unity transmission
(negative measured attenuation) can now enter firmware fits. For older captures whose candidate counts
disagree, valid points are labeled **fit membership unavailable**, without invented tail membership.
Nonpositive/undefined normalizations remain in the raw panels and record table. Probe points with a changing companion are diagnostic measurements, not
additional comparable sweep points. Captured fit membership, coefficients, RMS, and acceptance are unchanged.

In [ ]:
def calibration_display_data(dataset, context):
    """Add display-only normalization for excluded sweep points; leave firmware support unchanged.

    The driver retains signed attenuation for valid readings. This display also normalizes
    positive unsaturated low-SNR readings, without changing firmware fitting membership.
    Error bars include the same reference and bridge uncertainty, not independent fit weights.
    """
    # The driver infers membership only when the candidate prefix matches the captured fit count.
    frame = pd.DataFrame.from_records(dataset.derived())
    frame['display_db'] = np.nan
    frame['display_db_err'] = np.nan
    for physical in ('dac1', 'dac2'):
        meta = dataset._physical_meta(physical)
        rec = dataset.physical(physical).records
        if meta is None or not meta.get('reference_valid'):
            continue
        ref = dataset._record_by_index(rec, meta['reference_record'])
        if ref is None or ref.classification != 'ok' or ref.signal_mv <= 0 or ref.signal_err_mv <= 0:
            continue
        scales, variances = {0: 1.}, {0: 0.}
        for bridge in dataset._accepted_bridges(physical, rec):
            origin, dest = bridge['from_segment'], bridge['to_segment']
            if origin in scales:
                scales[dest] = scales[origin]*bridge['ratio']
                variances[dest] = variances[origin]+bridge['ratio_rel_var']
        for i, row in frame.loc[frame.physical.eq(physical)].iterrows():
            if row.event != 'point' or row.classification not in ('ok', 'below_snr') or row.segment not in scales:
                continue
            if row.signal_mv <= 0 or not np.isfinite(row.signal_err_mv) or row.signal_err_mv < 0:
                continue
            scaled = row.signal_mv/scales[row.segment]
            relative_var = (row.signal_err_mv/row.signal_mv)**2 + variances[row.segment]
            frame.loc[i, ['scaled_signal_mv', 'scaled_signal_err_mv']] = scaled, scaled*np.sqrt(relative_var)
            frame.loc[i, 'display_db'] = -10*np.log10(scaled/ref.signal_mv)
            frame.loc[i, 'display_db_err'] = 10/np.log(10)*np.sqrt(relative_var+(ref.signal_err_mv/ref.signal_mv)**2)
    return frame


def plot_calibration(dataset, context, physical, title=''):
    """Four panels with a flush, quarter-height residual panel. Pure offline display."""
    frame = calibration_display_data(dataset, context)
    p = frame.loc[frame.physical.eq(physical)].copy()
    if p.empty:
        print(f'{physical}: no retained records ({title}).')
        return None
    coeff = captured_coefficients(dataset, context).get(physical)
    fits = next((item['fits'] for item in dataset.meta if 'fits' in item), {})
    fit = fits.get(physical)
    fig = plt.figure(figsize=(12, 11))
    outer = fig.add_gridspec(3, 1, height_ratios=[1, 1, 2], hspace=.26)
    raw = fig.add_subplot(outer[0])
    scaled = fig.add_subplot(outer[1], sharex=raw)
    lower = outer[2].subgridspec(2, 1, height_ratios=[4, 1], hspace=0)
    curve_ax = fig.add_subplot(lower[0], sharex=raw)
    resid_ax = fig.add_subplot(lower[1], sharex=raw)
    axes = [raw, scaled, curve_ax, resid_ax]
    for classification, color in hspcb._ATTEN_CAL_CLASSIFICATION_COLORS.items():
        for event, marker in hspcb._ATTEN_CAL_EVENT_MARKERS.items():
            group = p.loc[p.classification.eq(classification) & p.event.eq(event)]
            if group.empty:
                continue
            raw.errorbar(group.sweep_mv, group.signal_mv, yerr=group.signal_err_mv.clip(lower=0),
                         fmt=marker, color=color, ms=3, elinewidth=.5, label=f'{event}: {classification}')
            # Variable-companion probes are diagnostics, not comparable sweep transmission.
            if event == 'point':
                good = group.loc[np.isfinite(group.scaled_signal_mv)]
                scaled.errorbar(good.sweep_mv, good.scaled_signal_mv, yerr=good.scaled_signal_err_mv,
                                fmt=marker, color=color, ms=3, elinewidth=.5)
    for role, indices in dataset._role_record_indices(physical).items():
        selected = p.loc[p.record.isin(indices)]
        for ax, col in ((raw, 'signal_mv'), (scaled, 'scaled_signal_mv')):
            ax.scatter(selected.sweep_mv, selected[col], marker=hspcb._ATTEN_CAL_ROLE_MARKERS[role],
                       facecolors='none', edgecolors=hspcb._ATTEN_CAL_ROLE_COLORS[role], s=65, label=role.replace('_', ' '))
    raw.set(ylabel='PD net (mV)', title='Raw dark-subtracted signal')
    scaled.set(ylabel='Scaled PD (mV)', title='Bridge-scaled signal')
    scaled.set_yscale('symlog', linthresh=.01)
    raw.legend(fontsize=7, ncol=3, loc='best')
    scaled.legend(fontsize=7, ncol=3, loc='best')
    measured = p.loc[np.isfinite(p.display_db)]
    if coeff is not None:
        grid = np.linspace(0, 3300, 3301)
        curve = hspcb._atten_db_from_coeff(coeff, grid)
        limit = float(fit.max_calibrated_db or 0)
        cross = np.flatnonzero(curve >= limit) if limit > 0 else np.array([], dtype=int)
        boundary = None
        if len(cross):
            k = int(cross[0])
            boundary = float(np.interp(limit, curve[max(0, k-1):k+1], grid[max(0, k-1):k+1]))
            grid = np.unique(np.r_[grid, boundary])
            curve = hspcb._atten_db_from_coeff(coeff, grid)
        inside = grid <= boundary if boundary is not None else np.zeros(len(grid), dtype=bool)
        curve_ax.plot(grid[inside], curve[inside], color='black', label='Captured firmware fit')
        curve_ax.plot(grid[~inside], curve[~inside], '--', color='black', alpha=.6, label='Rough continuation')
        if boundary is not None:
            curve_ax.axhline(limit, color='tab:purple', ls=':', lw=.8, label=f'Calibrated limit: {limit:g} dB')
            curve_ax.annotate(f'Rough region from {boundary:.1f} mV', (boundary, .98),
                              xycoords=('data', 'axes fraction'), xytext=(-5, -2), textcoords='offset points',
                              ha='right', va='top', color='tab:purple', fontsize=8)
            for ax in (curve_ax, resid_ax):
                ax.axvline(boundary, color='tab:purple', ls=':', lw=.8)
        support_known = int(fit.points or 0) > 0 and int(p.included.sum()) == int(fit.points)
        if support_known:
            in_range = measured.included & (measured.db.astype('float32') <= np.float32(limit))
            masks = [(in_range, 'o', 'tab:blue', 'Fit support in range'),
                     (measured.included & ~in_range, 'D', 'tab:orange', 'Fit support above limit'),
                     (~measured.included, 'x', '.5', 'Excluded')]
            tail = measured.loc[~measured.included & (measured.sweep_mv > measured.loc[measured.included, 'sweep_mv'].max())]
        else:
            # Historical count mismatches do not establish which valid points were excluded.
            excluded = measured.classification.eq('below_snr')
            masks = [(~excluded, 'o', 'tab:blue', 'Valid sweep; fit membership unavailable'),
                     (excluded, 'x', '.5', 'Excluded')]
            tail = measured.iloc[:0]
        if not tail.empty:
            for ax in (curve_ax, resid_ax):
                ax.axvspan(tail.sweep_mv.min(), 3300, color='.5', alpha=.10,
                           label='Excluded tail data' if ax is curve_ax else None)
        for mask, marker, color, label in masks:
            g = measured.loc[mask]
            if g.empty:
                continue
            curve_ax.errorbar(g.sweep_mv, g.display_db, yerr=g.display_db_err, fmt=marker,
                              color=color, ms=4, elinewidth=.6, label=label)
            residual = hspcb._atten_db_from_coeff(coeff, g.sweep_mv)-g.display_db
            resid_ax.errorbar(g.sweep_mv, residual, yerr=g.display_db_err,
                              fmt=marker, color=color, ms=3, elinewidth=.5)
        terms = max((i+1 for i, c in enumerate(coeff[4]) if c != 0), default=0)
        curve_ax.set_title(f'Captured fit: {terms} correction terms; accepted={fit.accepted}; firmware RMS={fit.rms_db:g} dB')
    else:
        curve_ax.errorbar(measured.sweep_mv, measured.display_db, yerr=measured.display_db_err,
                          fmt='x', color='.5', label='Excluded / no valid captured fit')
        curve_ax.set_title('No valid captured firmware fit; raw acquisition retained')
    for segment, group in p.loc[p.event.eq('point')].groupby('segment', sort=True):
        lo, hi = group.sweep_mv.min(), group.sweep_mv.max()
        if segment != 0:
            for ax in axes:
                ax.axvline(lo, color='.6', lw=.6, alpha=.35)
        curve_ax.text((lo+hi)/2, .03, f'segment {segment}', transform=curve_ax.get_xaxis_transform(),
                      ha='center', va='bottom', color='.5', fontsize=7)
    curve_ax.set_ylabel('Attenuation (dB)')
    curve_ax.legend(fontsize=7, loc='upper left')
    resid_ax.axhline(0, color='black', lw=.6)
    resid_ax.set(xlabel='DAC output (mV)', ylabel='Fit − data\n(dB)', xlim=(-30, 3330))
    for ax in axes[:-1]:
        ax.tick_params(labelbottom=False)
    # Avoid overlapping tick labels at the shared border without introducing a gap.
    resid_ax.yaxis.set_major_locator(plt.MaxNLocator(3, prune='both'))
    fig.suptitle(f'{physical} · {title}', fontsize=12)
    fig.subplots_adjust(top=.93, bottom=.07, left=.09, right=.98)
    return fig


In [ ]:
def review_calibration(dataset, context, path, name, *, show=True):
    """Summarize a captured result and draw the same offline figures after acquisition or during replay."""
    fits = next((m['fits'] for m in dataset.meta if 'fits' in m), {})
    states = []
    for item in dataset.meta:
        status = item.get('status')
        if status:
            state = status.get('state') if isinstance(status, dict) else status.state
            if state:
                states.append(state)
        if item.get('state'):
            states.append(item['state'])
    acquisition_state = next((state for state in states if state != 'complete'), 'complete' if states else 'unavailable')
    derived = pd.DataFrame.from_records(dataset.derived())
    rows = []
    for physical in ('dac1', 'dac2'):
        fit = fits.get(physical, hspcb.AttenuatorFitMetrics(valid=False))
        coeff = captured_coefficients(dataset, context).get(physical)
        curve = hspcb._atten_db_from_coeff(coeff, np.arange(3301)) if coeff else np.array([np.nan])
        p = derived.loc[derived.physical.eq(physical)]
        rows.append(dict(laser=name, file=str(path), physical=physical,
            condition=parameter(context, 'condition', 'historical / unspecified'),
            patch_id=parameter(context, 'patch_id', 'unspecified'),
            external_setup=parameter(context, 'external_setup', parameter(context, 'patch_notes', 'unspecified')),
            acquisition_state=acquisition_state,
            retrieval_error=next((m.get('retrieval_error') for m in dataset.meta if 'retrieval_error' in m), ''),
            records=len(p), valid=fit.valid, accepted=fit.accepted, points=fit.points,
            support_known=bool(fit.valid and fit.points and int(p.included.sum()) == int(fit.points)),
            max_calibrated_db=fit.max_calibrated_db, leakage_floor_db=fit.max_atten_db,
            rms_db=fit.rms_db, max_abs_db=fit.max_abs_db,
            monotonic=bool(np.all(np.isfinite(curve)) and np.all(np.diff(curve) >= -1e-5))))
    result = pd.DataFrame(rows)
    if show:
        print(f'{name} · {Path(path).name}; raw records remain available with dataset.to_dataframe().')
        display(result.drop(columns=['file', 'condition', 'patch_id', 'external_setup']))
        bridges = dataset.bridge_table()
        if not bridges.empty:
            display(bridges)
        for physical in ('dac1', 'dac2'):
            plot_calibration(dataset, context, physical, title=f'{name} · {Path(path).name}')
        plt.show()
    return result


In [ ]:
RUN_AUTOCAL = False
CAL_LASERS = list(LASERS)
CAL_DWELL_MS = 550
if RUN_AUTOCAL:
    pcb.stop_throughput('all')
    for name in LASERS:
        pcb.laser(name, value=0, autooff_s=0)
    for name in CAL_LASERS:
        context = snapshot(pcb, name)
        finished, archived, start_confirmed = False, False, False
        try:
            pcb.pd(CHANNEL)              # Explicitly power/refresh PD; use the already measured dark.
            state = pcb.atten_calibrate(name, output=OUTPUT, fiber=FIBER, dwell_ms=CAL_DWELL_MS, persist=False)
            start_confirmed = True
            while state.state == 'running':
                print(f'{name} {state.physical}: {state.complete_pct}%  {state.point}', end='\r')
                time.sleep(2)
                state = pcb.atten_calibrate()
            finished = True
            dataset = pcb.atten_calibration_data(physical='all')
            path = save_calibration(dataset, context, name)
            archived = True
            CALIBRATION_FILES[name] = path
            review_calibration(dataset, context, path, name)
        except BaseException as exc:
            if not archived:
                try:
                    dataset = pcb.atten_calibration_data(physical='all')
                    path = save_calibration(dataset, context, name, error=f'Partial retrieval; start confirmed={start_confirmed}. Requested source={name}; unconfirmed starts may retain the preceding acquisition. {type(exc).__name__}: {exc}')
                    CALIBRATION_FILES[name] = path
                    try:
                        review_calibration(dataset, context, path, name)
                    except Exception as display_error:
                        print(f'Partial capture saved; offline review failed: {display_error}')
                except Exception as retrieval_error:
                    save_tables(f'cal_error_{name}', context=context,
                        error=parameter_table(dict(acquisition_error=str(exc), retrieval_error=str(retrieval_error), data_available=False)))
            raise
        finally:
            try:
                if not finished:
                    pcb.atten_calibrate_stop()
            finally:
                pcb.laser(name, stop=True)


In [ ]:
# Add real saved captures here; no connection or new calibration is required.
REPLAY_CALIBRATIONS = dict(CALIBRATION_FILES)
# REPLAY_CALIBRATIONS['1028y'] = TOOLS / 'atten_noise_data/cal_1028y_20260918T003155_207983Z.npz'
cal_fit_rows = []
for name, path in REPLAY_CALIBRATIONS.items():
    dataset, context = load_calibration(path)
    cal_fit_rows.append(review_calibration(dataset, context, path, name))
cal_fit_table = pd.concat(cal_fit_rows, ignore_index=True) if cal_fit_rows else pd.DataFrame()
if cal_fit_table.empty:
    print('No calibration captures selected; acquire one or supply an existing NPZ path above.')


In [ ]:
ACCEPT_CALIBRATION_FILES = {}           # e.g. {'1028y': Path('...reviewed capture.npz')}
PERSIST_ACCEPTED_CALIBRATIONS = False
if PERSIST_ACCEPTED_CALIBRATIONS:
    for name, path in ACCEPT_CALIBRATION_FILES.items():
        dataset, context = load_calibration(path)
        if parameter(context, 'laser') != name:
            raise ValueError('Capture laser identity does not match target.')
        fits = next(item['fits'] for item in dataset.meta if 'fits' in item)
        reviewed = review_calibration(dataset, context, path, name, show=False)
        if reviewed.retrieval_error.fillna('').ne('').any() or not reviewed.acquisition_state.eq('complete').all():
            raise ValueError('Only a completed acquisition with no retrieval error can be persisted here.')
        if not all(fits[p].valid and fits[p].accepted for p in ('dac1', 'dac2')):
            raise ValueError('Both captured fits must be accepted before persisting this pair.')
        coefficients = {}
        for physical, c in captured_coefficients(dataset, context).items():
            coefficients[physical] = dict(fvoa_50pct_mv=c[0], slope_inv_fvoa_mv=c[1], max_atten_db=c[2],
                gain=c[3], correction_coeff=c[4], max_calibrated_db=c[5], rms_db=fits[physical].rms_db)
        before = pcb.atten_coeff(name)
        pcb.atten_coeff(name, coefficients['dac1'], coefficients['dac2'], persist=True)
        after = pcb.atten_coeff(name)
        display(pd.DataFrame([dict(physical=p, before_f50_mv=getattr(before, p).fvoa_50pct_mv,
            readback_f50_mv=getattr(after, p).fvoa_50pct_mv, readback_rms_db=getattr(after, p).rms_db)
            for p in ('dac1', 'dac2')]))
        save_tables(f'accepted_cal_{name}', source=np.array(str(path)), context=snapshot(pcb, name))
if not ACCEPT_CALIBRATION_FILES:
    print('No calibration files selected for persistence; acquisition results remain in RAM and archives.')


## Adjacent laser-current steps

Autocalibration is already complete. Query current settings again here; do not rerun it. The table starts with
minimum-autolevel and typical current for every laser. Typical means the presently positive current, otherwise
nominal, with its origin shown. Edit any row, choose subsets, or add a third operating point. Use previously
established FVOA settings that put the PD in range; keep both FVOAs fixed within an adjacent-current sequence.

Positive `laser(value=...)` spans threshold→nominal and quantizes to 0.1 mA; `value=0` means actual zero,
not threshold. A stored nonzero wavelength tune can alter the applied current/temperature: the notebook
records it and checks the applied setpoint. A flat 0.1 mA-resolution measured-current register does not prove
absence of finer or faster current noise. Temperature/readback cadence is separate from the optical stream.

In [ ]:
QUERY_CURRENT_SETUP = False
if QUERY_CURRENT_SETUP:
    current_rows = []
    for name in LASERS:
        settings, actual, drive = pcb.laser_settings(name), pcb.laser_status(name), pcb.atten(name)
        typical = actual.i_mA if actual.i_mA is not None and actual.i_mA > 0 else settings.nominal_current_ma
        for label, current, origin in [('minimum', settings.min_autolevel_current_ma, 'queried minimum autolevel'),
                ('typical', typical, 'present positive current' if actual.i_mA is not None and actual.i_mA > 0 else 'nominal fallback')]:
            current_rows.append(dict(laser=name, point=label, current_ma=current, origin=origin,
                threshold_ma=settings.threshold_current_ma, nominal_ma=settings.nominal_current_ma,
                measured_ma=actual.curr_meas_ma, tune_nm=settings.tune_nm,
                dac1_mv=drive.v1_mv, dac2_mv=drive.v2_mv, seconds=10.0))
    current_points = pd.DataFrame(current_rows)
    display(current_points)
# Edit in place; a third point is just another row. Durations are independent, not a total runtime budget.
# current_points.loc[0, ['current_ma', 'seconds']] = [14.6, 20.]
# current_points.loc[len(current_points)] = {**current_points.iloc[0].to_dict(), 'point': 'third', 'current_ma': 20.0}

In [ ]:
def current_fraction(settings, requested_ma):
    """Translate one reachable Maiman setpoint through the existing fractional command API."""
    if requested_ma == 0:
        return 0.
    low = math.ceil((settings.threshold_current_ma + 1e-9)*10)/10
    high = math.floor((settings.nominal_current_ma + 1e-9)*10)/10
    if not low <= requested_ma <= high or not np.isclose(requested_ma*10, round(requested_ma*10), atol=1e-7):
        raise ValueError(f'{requested_ma} mA is not a reachable positive 0.1 mA setpoint in [{low}, {high}].')
    return (requested_ma-settings.threshold_current_ma)/(settings.nominal_current_ma-settings.threshold_current_ma)

In [ ]:
RUN_CURRENT_STEPS = False
CURRENT_REPEATS = 2
if RUN_CURRENT_STEPS:
    for point in current_points.itertuples(index=False):
        settings = pcb.laser_settings(point.laser)
        center = round(point.current_ma*10)/10
        high = math.floor((settings.nominal_current_ma+1e-9)*10)/10
        neighbor = round((center+.1 if center+.1 <= high else center-.1)*10)/10
        for repeat in range(CURRENT_REPEATS):
            for step, requested in enumerate((center, neighbor, center)):
                fraction = current_fraction(settings, requested)
                monitor = open_stream(pcb, point.seconds, laser=point.laser)
                try:
                    for name in LASERS:
                        if name != point.laser:
                            pcb.laser(name, value=0, autooff_s=0)
                    pcb.atten(point.laser, value1_mv=point.dac1_mv, value2_mv=point.dac2_mv)
                    before_ms = time.time_ns()//1_000_000
                    pcb.laser(point.laser, value=fraction, autooff_s=0)
                    after_ms = time.time_ns()//1_000_000
                    actual = pcb.laser_status(point.laser)
                    context = snapshot(pcb, point.laser)
                    collect_trace(pcb, monitor, point.laser, point.seconds, 'current_steps', context=context,
                        extra=dict(point=point.point, repeat=repeat, step=step, requested_current_ma=requested,
                            applied_current_ma=actual.i_mA, measured_current_ma=actual.curr_meas_ma,
                            command_start_ms=before_ms, command_end_ms=after_ms,
                            current_match=np.isclose(actual.i_mA, requested, atol=.049) if actual.i_mA is not None else False))
                finally:
                    monitor.stop()
        pcb.laser(point.laser, value=0, autooff_s=0)

## Independent fixed-current holds and discontinuous revisits

This section stands on its own. Query the selected laser's current and FVOA settings, edit the visible
operating point, then capture. The archive includes its coefficients, dark, routing, and full settings. A 5 s, 200 s, or longer hold uses the same code with capacity
scaled to the requested duration. Warmup is observed in the trace, not guaranteed by a fixed delay.

The separate zero-current cell clears the auto-off deadline and keeps an already enabled driver ready. Return
half an hour later and run another hold: archives retain absolute board and host times, without joining the gap.
The explicit shutdown cell stops the driver and applies its TEC-off policy. No routine static-attenuator bypass
experiment is required; previous quiet bypass data are supporting evidence, not proof of the present noise floor.

In [ ]:
QUERY_HOLD_SETUP = False
HOLD_LASER = LASERS[0]
HOLD_SECONDS = 30.0
if QUERY_HOLD_SETUP:
    settings = pcb.laser_settings(HOLD_LASER)
    actual = pcb.laser_status(HOLD_LASER)
    drive = pcb.atten(HOLD_LASER)
    HOLD_CURRENT_MA = actual.i_mA if actual.i_mA is not None and actual.i_mA > 0 else settings.min_autolevel_current_ma
    HOLD_DAC_MV = (drive.v1_mv, drive.v2_mv)
    display(pd.DataFrame([dict(laser=HOLD_LASER, current_ma=HOLD_CURRENT_MA,
        dac1_mv=HOLD_DAC_MV[0], dac2_mv=HOLD_DAC_MV[1], seconds=HOLD_SECONDS)]))
# Edit HOLD_CURRENT_MA / HOLD_DAC_MV after querying; changing duration never reruns autocalibration.


In [ ]:
RUN_HOLD = False
if RUN_HOLD:
    settings = pcb.laser_settings(HOLD_LASER)
    fraction = current_fraction(settings, HOLD_CURRENT_MA)
    monitor = open_stream(pcb, HOLD_SECONDS, laser=HOLD_LASER)
    try:
        for name in LASERS:
            if name != HOLD_LASER:
                pcb.laser(name, value=0, autooff_s=0)
        pcb.atten(HOLD_LASER, value1_mv=HOLD_DAC_MV[0], value2_mv=HOLD_DAC_MV[1])
        command_start_ms = time.time_ns()//1_000_000
        pcb.laser(HOLD_LASER, value=fraction, autooff_s=0)
        command_end_ms = time.time_ns()//1_000_000
        context = snapshot(pcb, HOLD_LASER)
        hold_file = collect_trace(pcb, monitor, HOLD_LASER, HOLD_SECONDS, 'hold', context=context,
            extra=dict(requested_current_ma=HOLD_CURRENT_MA, command_start_ms=command_start_ms, command_end_ms=command_end_ms))
    finally:
        monitor.stop()

In [ ]:
ZERO_AND_REMAIN_ENABLED = False
if ZERO_AND_REMAIN_ENABLED:
    pcb.stop_throughput(CHANNEL)         # Manual stream ownership; no inherited autolevel owner.
    pcb.laser(HOLD_LASER, value=0, autooff_s=0)
    status = pcb.laser_status(HOLD_LASER)
    display(pd.DataFrame([dict(laser=HOLD_LASER, action='zero current; retain readiness',
        current_ma=status.i_mA, ready=status.ready, driver_started=status.op_started, tec_started=status.tec_started)]))


In [ ]:
FULL_SHUTDOWN_SELECTED = False
if FULL_SHUTDOWN_SELECTED:
    try:
        pcb.stop_throughput(CHANNEL)
    finally:
        pcb.laser(HOLD_LASER, stop=True)
    status = pcb.laser_status(HOLD_LASER)
    display(pd.DataFrame([dict(laser=HOLD_LASER, action='STOP', current_ma=status.i_mA,
        ready=status.ready, driver_started=status.op_started, tec_started=status.tec_started)]))


In [ ]:
RUN_AUTOOFF_EXERCISE = False
AUTOOFF_SECONDS = 5
if RUN_AUTOOFF_EXERCISE:
    pcb.stop_throughput(CHANNEL)
    settings = pcb.laser_settings(HOLD_LASER)
    pcb.atten(HOLD_LASER, value1_mv=HOLD_DAC_MV[0], value2_mv=HOLD_DAC_MV[1])
    command_start_ms = time.time_ns()//1_000_000
    pcb.laser(HOLD_LASER, value=current_fraction(settings, HOLD_CURRENT_MA), autooff_s=AUTOOFF_SECONDS)
    command_end_ms = time.time_ns()//1_000_000
    # Plain zero preserves the deadline; indefinite idle above explicitly uses autooff_s=0.
    pcb.laser(HOLD_LASER, value=0)
    observations, started = [], time.monotonic()
    try:
        while time.monotonic()-started < AUTOOFF_SECONDS+2:
            before_ms = time.time_ns()//1_000_000
            state, engineering = pcb.laser(HOLD_LASER), pcb.laser_status(HOLD_LASER)
            observations.append(dict(t_s=time.monotonic()-started, host_start_ms=before_ms,
                host_end_ms=time.time_ns()//1_000_000, off_in_s=state.off_in_s, current_ma=engineering.i_mA,
                powered=engineering.powered, ready=engineering.ready,
                driver_started=engineering.op_started, tec_started=engineering.tec_started))
            time.sleep(.5)
    finally:
        try:
            pcb.laser(HOLD_LASER, stop=True)
        finally:
            if observations:
                autooff_table = pd.DataFrame(observations)
                autooff_file = save_tables(f'autooff_{HOLD_LASER}', observations=autooff_table,
                    context=parameter_table(dict(laser=HOLD_LASER, channel=CHANNEL, experiment='autooff',
                        condition=CONDITION, build_label=BUILD_LABEL, patch_id=f'{OUTPUT}_to_{FIBER}',
                        external_setup=patches.loc[f'{OUTPUT}_to_{FIBER}', 'external_setup'],
                        external_loss_db=patches.loc[f'{OUTPUT}_to_{FIBER}', 'external_loss_db'],
                        external_loss_basis=patches.loc[f'{OUTPUT}_to_{FIBER}', 'external_loss_basis'],
                        requested_timeout_s=AUTOOFF_SECONDS, command_start_ms=command_start_ms, command_end_ms=command_end_ms)))
                CAPTURE_FILES.append(autooff_file)
                display(autooff_table)
                fig, axes = plt.subplots(2, 1, sharex=True, figsize=(10, 4), layout='constrained')
                for column in ('ready', 'driver_started', 'tec_started'):
                    axes[0].step(autooff_table.t_s, autooff_table[column].astype(int), where='post', label=column)
                axes[0].set_ylabel('Reported state'); axes[0].legend(fontsize=8)
                axes[1].plot(autooff_table.t_s, autooff_table.off_in_s, '.-', label='Reported time remaining')
                axes[1].set(xlabel='Seconds from polling start after zero-current command', ylabel='Seconds remaining')
                stopped = autooff_table.loc[~autooff_table.driver_started]
                print('STOP first observed at polling t = '+str(stopped.t_s.iloc[0])+' s' if not stopped.empty else 'STOP was not observed in this polling window.')
                print('Polling and command brackets bound timing; this is not an optical response measurement.')
                plt.show()
            else:
                print('No auto-off observations collected; no empty archive written.')


## Repeated levels, attenuation redistribution, and tail checks

Query each laser's existing settings before building this experiment. The default row order compares
25+35 → 35+25 → 25+25 → 25+35 dB at one current, then scans a physical FVOA through its calibrated boundary.
Edit operating points to suit the assembly and PD range. Raw records keep settling, overrange and dark data;
analysis selects stationary usable intervals explicitly. Calibrated boundaries come from queried coefficients,
not an assumed 55 dB for every device. Commands beyond the boundary probe a rough model and are labeled so.

In [ ]:
QUERY_NOISE_SETUP = False
if QUERY_NOISE_SETUP:
    noise_rows = []
    for name in LASERS:
        settings, actual, coefficients = pcb.laser_settings(name), pcb.laser_status(name), pcb.atten_coeff(name)
        current = actual.i_mA if actual.i_mA is not None and actual.i_mA > 0 else settings.min_autolevel_current_ma
        for label, a, b in [('redistribute_A',25.,35.), ('redistribute_B',35.,25.), ('brighter',25.,25.), ('redistribute_A_repeat',25.,35.)]:
            noise_rows.append(dict(laser=name, point=label, current_ma=current, dac1_db=a, dac2_db=b, seconds=30.))
        for physical in ('dac1', 'dac2'):
            limit = getattr(coefficients, physical).max_calibrated_db
            for offset in (-2.5, 0, 2.5, 5.):
                value = max(0., limit+offset)
                noise_rows.append(dict(laser=name, point=f'{physical}_limit{offset:+g}', current_ma=current,
                    dac1_db=value if physical == 'dac1' else 0., dac2_db=value if physical == 'dac2' else 0., seconds=30.))
    noise_points = pd.DataFrame(noise_rows)
    display(noise_points)

In [ ]:
RUN_NOISE_POINTS = False
if RUN_NOISE_POINTS:
    for point in noise_points.itertuples(index=False):
        settings = pcb.laser_settings(point.laser)
        fraction = current_fraction(settings, round(point.current_ma*10)/10)
        monitor = open_stream(pcb, point.seconds, laser=point.laser)
        try:
            for name in LASERS:
                if name != point.laser:
                    pcb.laser(name, value=0, autooff_s=0)
            command_start_ms = time.time_ns()//1_000_000
            applied = pcb.atten(point.laser, value1_db=point.dac1_db, value2_db=point.dac2_db)
            pcb.laser(point.laser, value=fraction, autooff_s=0)
            command_end_ms = time.time_ns()//1_000_000
            context = snapshot(pcb, point.laser)
            collect_trace(pcb, monitor, point.laser, point.seconds, 'noise', context=context,
                extra=dict(point=point.point, requested_dac1_db=point.dac1_db, requested_dac2_db=point.dac2_db,
                    applied_dac1_mv=applied.v1_mv, applied_dac2_mv=applied.v2_mv,
                    command_start_ms=command_start_ms, command_end_ms=command_end_ms))
        finally:
            monitor.stop()
    for name in LASERS:
        pcb.laser(name, value=0, autooff_s=0)

## Throughput, autolevel, and response

Run the installed policy and label the build. The editable table includes all three lasers and both outputs;
select rows matching the physical patch, then repeat after repatching. The collector/dashboard is the existing
driver implementation. Autolevel changes current/attenuation and owns laser shutdown. Manual commands disable
its adjustment loop but retain that ownership, which is why later manual sections stop it **before** source setup.

The 50 ms publication cadence is one fresh ADC conversion, not a continuous 50 ms integration. Missing records
are gaps. The 500 ms PD diagnostic window and configurable calibration averages are separate measurements.
The firmware's 400–1600 mV useful band, movements, and settling lag remain visible. A throughput peak is a
loose loopback reference; there is no unity target. Values depend on measured route losses and wavelength
response assumptions. An overrange reading is a nominal lower bound, not a precise calibration point.

In [ ]:
throughput_points = pd.DataFrame([dict(laser=name, output=f'{CHANNEL}_{out}', fiber=FIBER,
    initial_level=.5, seconds=60., patch_id=f'{CHANNEL}_{out}_to_{FIBER}') for name in LASERS for out in ('ao', 'fei')])
display(throughput_points)
RUN_THROUGHPUT = False
THROUGHPUT_ROW = 0
if RUN_THROUGHPUT:
    point = throughput_points.iloc[THROUGHPUT_ROW]
    display(pd.DataFrame([{**point.to_dict(), **patches.loc[point.patch_id].to_dict()}]))
    context = pd.concat([snapshot(pcb, point.laser, output=point.output, fiber=point.fiber), parameter_table({'run': point.to_dict()})])
    monitor = open_stream(pcb, point.seconds, laser=point.laser, fiber=point.fiber, output=point.output,
                          autolevel=True, initial_level=point.initial_level)
    throughput_file = collect_trace(pcb, monitor, point.laser, point.seconds, 'throughput', context=context,
                                   extra=dict(point=point.patch_id))


In [ ]:
# Optional live exploration; execute this cell, then the save/stop cell when finished.
# Enable an interactive backend with %matplotlib widget in a separate cell if desired.
START_LIVE_DASHBOARD = False
LIVE_LASER, LIVE_CAPACITY_SECONDS = LASERS[0], 600.
if START_LIVE_DASHBOARD:
    if 'tp_figure' in globals():
        plt.close(tp_figure)
    live_context = snapshot(pcb, LIVE_LASER)
    monitor = open_stream(pcb, LIVE_CAPACITY_SECONDS, laser=LIVE_LASER, autolevel=True, initial_level=.5)
    tp_figure, tp_animation = monitor.plot_live(channel=CHANNEL, max_points=600)
    display(tp_figure)
# tp_animation.pause()/resume() affect only display. Hardware and collection keep running.

In [ ]:
SAVE_AND_STOP_LIVE = False
if SAVE_AND_STOP_LIVE:
    try:
        monitor.stop()
    finally:
        samples = monitor.to_dataframe()
        samples['source_laser'], samples['experiment'] = LIVE_LASER, 'throughput_live'
        samples['condition'], samples['build_label'] = CONDITION, BUILD_LABEL
        path = save_tables(f'throughput_live_{LIVE_LASER}', samples=samples, context=live_context,
            outcome=parameter_table(dict(capacity=monitor.max_samples, capacity_reached=len(samples)>=monitor.max_samples)),
            source_reference=source_power_readings, nominal_reference=laser_nominal.reset_index())
        CAPTURE_FILES.append(path)
        review_capture(path)


## Offline analysis and cross-laser / environmental comparison

Select archives explicitly to compare room and chamber data, different laser currents, attenuation distributions,
and both firmware builds. Every file/interval remains separate, with its physical patch and external setup.
A bench pad and a later instrument are different configurations; unknown external loss stays unknown. Drift and RMS answer different questions;
plots retain the full trace as well as stationary-segment statistics. The averaging plot compares measured
block-mean scatter with an independent-sample prediction; adjacent-block correlation and Allan deviation help
identify when that prediction fails. Shared calibration uncertainty never shrinks as 1/√N here.

The PSD uses contiguous samples and their median interval; timing jitter is reported. It is not a frequency
response correction for arbitrary irregular sampling. Temporal PD scatter is not `tp_err`. The latter includes
source/curve calibration terms and a static independent-drive electrical-noise assumption (10 mV per FVOA).
Do not equate residual fit RMS with electrical temporal noise or assume all excess noise comes from FVOAs.


In [ ]:
ANALYSIS_FILES = list(dict.fromkeys(CAPTURE_FILES))   # Add archived room/chamber/build files explicitly.
SETTLE_S, MIN_STATIONARY_S = 1., 2.
PLOT_REPLAY_TRACES = False             # Acquisition already displayed traces; True redraws selected archives.
analysis_parts = {key: [] for key in ('audit', 'noise', 'spectra', 'averaging', 'throughput')}
for path in ANALYSIS_FILES:
    result = review_capture(path, settle_s=SETTLE_S, minimum_s=MIN_STATIONARY_S,
                            show=PLOT_REPLAY_TRACES, detailed=PLOT_REPLAY_TRACES)
    for key, frame in result.items():
        if not frame.empty:
            analysis_parts[key].append(frame)
analysis_tables = {key: pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
                   for key, parts in analysis_parts.items()}
noise_summary, spectra, averaging = (analysis_tables[key] for key in ('noise', 'spectra', 'averaging'))
throughput_summary, capture_audit = analysis_tables['throughput'], analysis_tables['audit']
if capture_audit.empty:
    print('No sampled captures selected. Add NPZ paths above; this cell does not acquire data.')
else:
    display(capture_audit, noise_summary, throughput_summary)
if not noise_summary.empty:
    # Keep each file and interval visible. Any pooling across configurations must be an explicit later decision.
    display(noise_summary[['file', 'interval', 'laser', 'point', 'condition', 'build', 'patch_id',
        'external_setup', 'external_loss_db', 'external_loss_basis', 'current_ma', 'mean_mv', 'rms_mv', 'drift_mv_per_s']])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout='constrained')
    for (file, interval), group in spectra.groupby(['file', 'interval'], sort=False):
        positive = group.f_hz.gt(0) & group.psd_mv2_per_hz.gt(0)
        axes[0].loglog(group.loc[positive, 'f_hz'], np.sqrt(group.loc[positive, 'psd_mv2_per_hz']), label=f'{Path(file).stem}:{interval}')
    for (file, interval), group in (averaging.groupby(['file', 'interval'], sort=False) if not averaging.empty else []):
        line, = axes[1].loglog(group.tau_s, group.rms_mean_mv, '.-', label=f'{Path(file).stem}:{interval}')
        axes[1].loglog(group.tau_s, group.iid_rms_mean_mv, ':', color=line.get_color(), alpha=.6)
    axes[0].set(xlabel='Frequency (Hz)', ylabel='PD amplitude density (mV/√Hz)')
    axes[1].set(xlabel='Block duration (s)', ylabel='Block-mean RMS (mV)')
    for ax in axes:
        if ax.lines:
            ax.legend(fontsize=6)
    plt.show()


### Shot-noise scale and normalization assumptions

For mean ADC-input signal V and effective transimpedance G (including the divider), the elementary photocurrent
shot-noise scale is √(2qVGB). Here B is an **equivalent noise bandwidth**, not the detector's quoted −3 dB
bandwidth or the telemetry Nyquist frequency. Enter measured/justified bandwidths below; scenarios remain
scenarios. A nearby measured dark captures detector/electronics noise already present. Excess after quadrature
subtraction is descriptive: drift, laser, FVOA, detector and acquisition effects are not separated by this number.
Per-laser responsivity has no separate current command API; inspect the channel setting/wavelength assumptions
and keep externally measured values in the tables instead of silently rewriting calibration during analysis.

In [ ]:
ENBW_SCENARIOS_HZ = [5., 10., 20.]      # Illustrative scenarios, not measured ADC/detector response.
# Optional per-file adjacent dark overrides; otherwise use that capture's active dark RMS.
DARK_RMS_OVERRIDES = {}
shot_rows = []
for row in noise_summary.itertuples(index=False):
    for bandwidth in ENBW_SCENARIOS_HZ:
        dark_rms = DARK_RMS_OVERRIDES.get(row.file, row.active_dark_rms_mv)
        shot_mv = 1000*np.sqrt(2*1.602176634e-19*max(0,row.mean_mv/1000)*row.effective_gain_v_per_a*bandwidth)
        excess_variance = row.rms_mv**2-dark_rms**2-shot_mv**2
        shot_rows.append(dict(file=row.file, laser=row.laser, interval=row.interval, condition=row.condition,
            enbw_scenario_hz=bandwidth, measured_rms_mv=row.rms_mv, dark_rms_mv=dark_rms, shot_rms_mv=shot_mv,
            excess_variance_mv2=excess_variance, excess_rms_mv=np.sqrt(max(0,excess_variance)) if np.isfinite(excess_variance) else np.nan))
shot_table = pd.DataFrame(shot_rows)
if shot_table.empty:
    print('No stationary measurements for shot-noise scenarios. Select captures and run the analysis above.')
else:
    display(shot_table)
SAVE_ANALYSIS = False
if SAVE_ANALYSIS:
    if noise_summary.empty and throughput_summary.empty:
        print('No analysis results to save; no empty archive written.')
    else:
        save_tables('analysis', noise_summary=noise_summary, throughput_summary=throughput_summary,
            audit=capture_audit, spectra=spectra, averaging=averaging, shot_scenarios=shot_table,
            source_reference=source_power_readings)


### Commanded response and repeatability

Select a capture and inspect a contiguous transition with separate baseline and final windows. The plots show
the selected windows and retain raw points. Crossing times are seconds from the first retained sample; the
10–90% interval is their difference. Command host-time brackets are recorded, not treated as an oscilloscope
trigger. Missing levels, gaps at a crossing, or no observed step leave the result unresolved. Rising and falling
steps use the same signed normalization. At 20 Hz, sub-sample response is unresolved; the separate scope notebook
measures simultaneous drive and PD steps when this stream is too slow.


In [ ]:
RESPONSE_FILE = None
RESPONSE_WINDOW_S = (0., 10.)
BASELINE_WINDOW_S = (0., 1.)
FINAL_WINDOW_S = (8., 10.)
response_result = pd.DataFrame()
if RESPONSE_FILE is None:
    print('Select a response capture and baseline/final windows above; no response measurement selected.')
else:
    response = load_tables(RESPONSE_FILE)['samples']
    if response.empty:
        print('Selected capture contains no samples; response is unresolved.')
    else:
        rt = (response.t_ms-response.t_ms.iloc[0])/1000
        valid = np.isfinite(response.pd_net_mv) & np.isfinite(response.pd_mv) & response.pd_mv.lt(hspcb.PD_ADC_USABLE_MV)
        select = rt.between(*RESPONSE_WINDOW_S)
        baseline = response.loc[valid & select & rt.between(*BASELINE_WINDOW_S), 'pd_net_mv'].median()
        final = response.loc[valid & select & rt.between(*FINAL_WINDOW_S), 'pd_net_mv'].median()
        relative = (response.pd_net_mv-baseline)/(final-baseline) if np.isfinite(final-baseline) and final != baseline else pd.Series(np.nan, index=response.index)
        dt = rt.diff(); cadence = dt[dt > 0].median()
        adjacent = valid & valid.shift(fill_value=False) & dt.gt(0) & dt.le(1.5*cadence)
        crossing, search_from = {}, max(RESPONSE_WINDOW_S[0], BASELINE_WINDOW_S[1])
        for fraction in (.1, .5, .9):
            hits = rt.loc[select & rt.ge(search_from) & adjacent & relative.ge(fraction) & relative.shift().lt(fraction)]
            crossing[fraction] = float(hits.iloc[0]) if len(hits) else np.nan
            if len(hits):
                search_from = crossing[fraction]
        resolved = all(np.isfinite(list(crossing.values())))
        response_result = pd.DataFrame([dict(file=str(RESPONSE_FILE), baseline_mv=baseline, final_mv=final,
            t10_s=crossing[.1], t50_s=crossing[.5], t90_s=crossing[.9],
            transition_10_90_s=crossing[.9]-crossing[.1] if resolved else np.nan,
            sample_interval_s=cadence, result='resolved at sampling cadence' if resolved else 'unresolved crossings/windows')])
        display(response_result)
        brackets = [c for c in ('command_start_ms', 'command_end_ms') if c in response]
        if brackets:
            display(response[brackets].drop_duplicates())
        fig, axes = plt.subplots(2, 1, sharex=True, figsize=(11, 5), layout='constrained')
        axes[0].plot(rt[select], response.loc[select, 'pd_net_mv'], '.-', ms=3)
        axes[1].plot(rt[select], relative[select], '.-', ms=3)
        for ax in axes:
            ax.axvspan(*BASELINE_WINDOW_S, color='tab:green', alpha=.15, label='Baseline window')
            ax.axvspan(*FINAL_WINDOW_S, color='tab:orange', alpha=.15, label='Final window')
            ax.set_xlim(*RESPONSE_WINDOW_S)
        for fraction, when in crossing.items():
            axes[1].axhline(fraction, color='.5', lw=.6, ls=':')
            if np.isfinite(when):
                axes[1].plot(when, relative.loc[rt.eq(when)].iloc[0], 'o', label=f'{fraction:.0%}: {when:g} s')
        axes[0].set_ylabel('PD net (mV)'); axes[0].legend(fontsize=8)
        axes[1].set(xlabel='Seconds from first retained sample', ylabel='Fraction of observed step')
        axes[1].legend(fontsize=8)
        plt.show()


## Commissioning record

Archive the selected measurements and annotate conclusions in this table: what the optical routing matrix
established, assembled losses adopted, fit acceptance/range, current-step repeatability, warmup/drift/noise,
response limits, and throughput references. Compare all three lasers and both environments, with the active
calibration and build attached. Leave unmeasured items explicitly unmeasured; no arbitrary acceptance limits
are supplied by the notebook. Later HISPEC acquisition code or a future driver can adopt procedures after
bench validation without importing this notebook as a hidden dependency.

In [ ]:
# Rebuild factual evidence from archives; adopted values and scientific conclusions remain editable below.
evidence_rows = []
for path in dict.fromkeys(CAPTURE_FILES):
    saved = load_tables(path)
    ctx = saved.get('context', parameter_table({}))
    samples = saved.get('samples', pd.DataFrame())
    observations = saved.get('observations', pd.DataFrame())
    experiment = str(samples.experiment.iloc[0]) if len(samples) else parameter(ctx, 'experiment', 'unmeasured')
    evidence_rows.append(dict(capture=str(path), function=experiment, laser=parameter(ctx, 'laser', 'unspecified'),
        condition=parameter(ctx, 'condition', 'unspecified'), build=parameter(ctx, 'build_label', 'unspecified'),
        patch_id=parameter(ctx, 'patch_id', 'unspecified'),
        external_setup=parameter(ctx, 'external_setup', parameter(ctx, 'patch_notes', 'unspecified')),
        samples=len(samples), observations=len(observations),
        complete=parameter(saved.get('outcome', parameter_table({})), 'complete', 'not recorded')))
for name, path in CALIBRATION_FILES.items():
    dataset, ctx = load_calibration(path)
    summary = review_calibration(dataset, ctx, path, name, show=False)
    evidence_rows.append(dict(capture=str(path), function='autocalibration', laser=name,
        condition=parameter(ctx, 'condition', 'unspecified'), build=parameter(ctx, 'build_label', 'unspecified'),
        patch_id=parameter(ctx, 'patch_id', 'unspecified'), external_setup=parameter(ctx, 'external_setup', 'unspecified'),
        samples=len(dataset.records), observations=0, complete=summary.acquisition_state.iloc[0],
        firmware_accepted=bool(summary.accepted.all())))
for name, group in source_power_readings.groupby('laser', sort=False):
    evidence_rows.append(dict(capture='inline source_power_readings', function='external power reference', laser=name,
        observations=len(group), external_setup='S154C + static attenuator + measurement fiber of unknown loss'))
evidence_summary = pd.DataFrame(evidence_rows)
display(evidence_summary)
commissioning_record = globals().get('commissioning_record', pd.DataFrame(columns=[
    'condition', 'build', 'laser', 'patch_id', 'external_setup', 'function', 'capture',
    'observation', 'adopted_value', 'units', 'remaining_question']))
# Add rows using capture paths above; do not substitute firmware fit acceptance for your bench conclusion.
display(commissioning_record)
SAVE_COMMISSIONING_RECORD = False
if SAVE_COMMISSIONING_RECORD:
    if commissioning_record.empty:
        print('No conclusions entered; no empty commissioning record saved.')
    else:
        save_tables('commissioning_record', evidence=evidence_summary, conclusions=commissioning_record,
                    source_reference=source_power_readings)


## Offline replay and interpretation

Run imports, reference tables, archive/statistical definitions, and the desired replay cells without executing
connection or acquisition cells. Set explicit NPZ paths for calibration replay or `ANALYSIS_FILES`; use
`review_capture(path, detailed=True)` to inspect a saved trace and slow engineering telemetry. NPZ archives
retain raw data and DataFrame-compatible tables, including source references for new optical captures.

Acquisition plots and replay use the same analysis. Settling cuts never remove saved samples. A saved file,
acknowledged command, or firmware-accepted fit is evidence to inspect, not an automatic commissioning verdict.
Record outstanding measurements and physical uncertainties in the commissioning table. Preserve unknown external
and meter-cable losses when transferring procedures to HISPEC acquisition code or a future wrapper.
